# 16 — E4 data + baselines  (auto-generado por build_notebook_16_e4_data.py)

**Corredor de validación E4** (`empresaid=4`) — tercer corredor para validez
externa. Este kernel CPU hace TODO en un solo notebook:

1. **Preprocessing E4** — pipeline de Fase 2 (mirror de NB04) sobre
   `clean_gps.parquet` (que ya contiene la empresa 4), produciendo
   `headways_E4.parquet` (+ `cleaned_gps_E4`, `headway_null_buckets_E4`).
2. **Baselines E4** — B0–B4 + B5_XGB (mirror de NB10) a horizontes
   h ∈ {1, 3, 5, 10}, escribiendo `baselines_E4_results_multih.csv`.

E4 usa la estrategia de centerline `"single"` (gated por
`centerline_strategy_for(4)`, NO hardcodeado) y `has_heading=True`. Los
notebooks NB04–NB13 son artefactos congelados: E4 REUSA la librería vía este
builder nuevo sin tocarlos.

In [ ]:

import polars as pl
import numpy as np
from pathlib import Path
import os

# Locate clean_gps.parquet under /kaggle/input (or local working directory).
candidates = list(Path("/kaggle/input").rglob("clean_gps.parquet")) if Path("/kaggle/input").exists() else []
if not candidates:
    candidates = list(Path(".").rglob("clean_gps.parquet"))
if not candidates:
    raise FileNotFoundError("clean_gps.parquet not found. Expected at /kaggle/input/**/clean_gps.parquet")
INPUT = candidates[0]
print(f"Input: {INPUT}")

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)
CSV_OUT = OUTPUT_DIR / "baselines_E4_results_multih.csv"
print(f"Output dir: {OUTPUT_DIR}")

# E4 ONLY — frozen corridors E2/E59 are NOT touched by this notebook.
EMPRESAS = [4]

## Module: preprocessing/config

Parámetros productivos congelados desde `docs/decisiones-headway-fase2.md §3`.
E4 está en `EMPRESA_CONFIG` (empresaid=4, has_heading=True, estrategia "single").

In [ ]:
"""Configuration and frozen parameters for the Fase 2 preprocessing pipeline.

All productive parameter values are locked to docs/decisiones-headway-fase2.md §3.
Changing any value requires updating that document first (versioned decision),
then updating the literal here. The freeze-assertion test in
tests/preprocessing/test_config.py encodes this contract as executable checks.
"""
import math
from dataclasses import dataclass
from typing import Literal, Mapping

# ---------------------------------------------------------------------------
# Coordinate constants — local flat-Earth at Arequipa (-16.4°)
# ---------------------------------------------------------------------------

LAT_DEG_M: float = 111_000.0
LON_DEG_M: float = 111_000.0 * math.cos(math.radians(-16.4))

# ---------------------------------------------------------------------------
# Quality thresholds (decisiones-limpieza-fase2 §2 rows 4-5)
# ---------------------------------------------------------------------------

MAX_PLAUSIBLE_SPEED_KMH: float = 80.0
MAX_PLAUSIBLE_JUMP_M: float = 500.0

# ---------------------------------------------------------------------------
# Trip segmentation (decisión §3.3 of decisiones-limpieza-fase2)
# ---------------------------------------------------------------------------

GAP_CUT_SECONDS: int = 30 * 60         # 30-minute gap between consecutive pings
TERMINAL_BAND_M: float = 200.0         # within X m of s_min / s_max → terminal candidate
TERMINAL_DWELL_SECONDS: int = 5 * 60   # stopped > 5 min near a terminal → cut
TERMINAL_MAX_SPEED_KMH: float = 5.0    # stopped threshold for terminal-dwell detection


# ---------------------------------------------------------------------------
# Frozen productive parameters
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class ProductiveParams:
    """Frozen contract — every field mirrors docs/decisiones-headway-fase2.md §3.

    Changing a value requires updating that document FIRST (versioned), then
    this file. The freeze-assertion test in test_config.py turns this into
    executable code.
    """

    grid_seconds: int = 60
    min_speed_for_centerline_kmh: float = 10.0
    centerline_latlon_quantile_lo: float = 0.005
    centerline_latlon_quantile_hi: float = 0.995
    centerline_n_bins: int = 50
    centerline_trim_pct: float = 0.025
    centerline_smooth_win: int = 5
    lateral_offset_threshold_m: float = 300.0
    direction_smooth_win: int = 5
    min_buses_per_snapshot: int = 2
    # Max staleness in minutes for a historical crossing to count as a real
    # trailing pair. Older crossings → emit delta_t_min = NULL. Bound exists
    # because multi-filar corridors (e.g. E2 in Arequipa) project unrelated
    # buses to the same s; without this bound, np.searchsorted finds ancient
    # crossings and reports them as valid headways. See decisiones-headway-fase2 §3.
    max_interpolation_lookback_minutes: float = 30.0
    # Lateral distance threshold (meters) between bus_front and bus_back to
    # consider them on the same track. Pairs with |lateral_m_front -
    # lateral_m_back| > threshold are filtered out as cross-street pairs.
    #
    # DEFAULT: float('inf') — filter is OFF by default (no-op).
    # Rationale: Kaggle 04b v4 Figure 7 (2026-05-21) showed a monotonically-
    # decreasing |lateral_delta| distribution for E2/E59 with no bimodal valley.
    # A calibration threshold cannot be meaningfully chosen from this shape.
    # Root cause is upstream (centerline + projection per direction), addressed
    # by Option D SDD (multi-filar-direction-balanced-centerline).
    #
    # Opt-in: set EmpresaConfig.lateral_pair_threshold_m_override to a finite
    # value for any empresa where the filter should be active.
    # See decisiones-headway-fase2 §7.0b.1.
    #
    # Per-empresa override available via EmpresaConfig.lateral_pair_threshold_m_override.
    lateral_pair_threshold_m: float = float('inf')
    # Centerline strategy: "single" (default, single-pass PCA over all pings)
    # or "two-pass" (per-direction PCA for multi-filar corridors).
    # Overridden per empresa via EmpresaConfig.centerline_strategy_override.
    # See decisiones-headway-fase2 §8, R-CFG1.
    centerline_strategy: Literal["single", "two-pass"] = "single"
    # Minimum ping count per direction subset to attempt pass-2 PCA.
    # Below this threshold, build_centerline_per_direction falls back to the
    # single-pass centerline. See R-CL1, R-CFG1.
    centerline_min_pings_per_direction: int = 1_000


PRODUCTIVE_PARAMS = ProductiveParams()


# ---------------------------------------------------------------------------
# Per-empresa configuration
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class EmpresaConfig:
    """Per-empresa settings.

    has_heading: E2/E4 report a `direccion` field usable as cross-check;
                 E58/E59 do not.
    centerline_sample_cap: maximum pings used to build the centerline.
    lateral_offset_threshold_m_override: when set, overrides
        PRODUCTIVE_PARAMS.lateral_offset_threshold_m for this empresa.
        Used for Caveat 3 monitoring (see decisiones-headway-fase2 §4).
    """

    empresaid: int
    has_heading: bool
    centerline_sample_cap: int = 50_000
    lateral_offset_threshold_m_override: float | None = None
    # Per-empresa override for the lateral pair filter threshold (meters).
    # When set, overrides PRODUCTIVE_PARAMS.lateral_pair_threshold_m for this
    # empresa. Used after Kaggle calibration of the |lateral_delta| histogram.
    lateral_pair_threshold_m_override: float | None = None
    # Per-empresa override for the centerline strategy.
    # When set, overrides PRODUCTIVE_PARAMS.centerline_strategy for this empresa.
    # E2 and E59 are set to "two-pass" (multi-filar corridor, Option D SDD).
    centerline_strategy_override: Literal["single", "two-pass"] | None = None


EMPRESA_CONFIG: Mapping[int, EmpresaConfig] = {
    2:  EmpresaConfig(empresaid=2,  has_heading=True,  centerline_strategy_override="two-pass"),
    59: EmpresaConfig(empresaid=59, has_heading=False, centerline_strategy_override="two-pass"),
    # E4 — third (validation) corridor for external validity (added 2026-06-22).
    # has_heading=True: E4 reports a `direccion` field (like E2).
    # centerline_strategy_override left None (→ "single", global default) PROVISIONALLY:
    # E2/E59 were set to "two-pass" only after Kaggle NB04 calibration evidence
    # (multi-filar projection). E4 has no such evidence yet — run NB04 for E4,
    # inspect the stale-crossing / headway sanity outputs, and flip to "two-pass"
    # here (1-line change) if it shows the same multi-filar behaviour.
    4:  EmpresaConfig(empresaid=4,  has_heading=True),
}


# ---------------------------------------------------------------------------
# Per-direction sort key calibration (SDD dir1-pair-ordering-h7)
# ---------------------------------------------------------------------------
# Empirically calibrated via observational evidence from Kaggle NB04 v7
# bucket analysis (obs #126). See sdd/dir1-pair-ordering-h7/apply-progress
# for full calibration evidence and cross-references.
#
# Set to +1 because the dir=+1 per-direction centerline has `s` inverse to
# physical direction of motion for empresas E2 and E59 (multi-filar corridor,
# two-pass PCA). Evidence: dir+1 yields 83.5%/92.6% stale-crossing rate;
# dir-1 yields ~70% success rate.
#
# CALIBRATED_INVERTED_DIRECTION: the direction value whose sort key is -s
# (negated arc-length) so that ascending sort places the physically-front
# bus first. For the other direction, sort key == s (canonical ascending).
CALIBRATED_INVERTED_DIRECTION: Literal[1, -1] = 1


def centerline_strategy_for(empresaid: int) -> str:
    """Return the effective centerline strategy for a given empresa.

    Checks EmpresaConfig.centerline_strategy_override first; falls back to
    PRODUCTIVE_PARAMS.centerline_strategy. Returns the global default for
    empresas not in EMPRESA_CONFIG (graceful missing-key handling).

    R-CFG1: E2 and E59 return "two-pass"; all others return "single".
    """
    cfg = EMPRESA_CONFIG.get(empresaid)
    if cfg is not None and cfg.centerline_strategy_override is not None:
        return cfg.centerline_strategy_override
    return PRODUCTIVE_PARAMS.centerline_strategy


def lateral_threshold_for(empresaid: int) -> float:
    """Return the effective lateral offset threshold for a given empresa.

    Checks EmpresaConfig.lateral_offset_threshold_m_override first; falls
    back to PRODUCTIVE_PARAMS.lateral_offset_threshold_m (Caveat 3 hook).
    Returns the global default for empresas not in EMPRESA_CONFIG (graceful
    missing-key handling, mirroring the other resolvers).
    """
    cfg = EMPRESA_CONFIG.get(empresaid)
    if cfg is not None and cfg.lateral_offset_threshold_m_override is not None:
        return cfg.lateral_offset_threshold_m_override
    return PRODUCTIVE_PARAMS.lateral_offset_threshold_m


def lateral_pair_threshold_for(empresaid: int) -> float:
    """Return the effective lateral pair filter threshold for a given empresa.

    Checks EmpresaConfig.lateral_pair_threshold_m_override first; falls back
    to PRODUCTIVE_PARAMS.lateral_pair_threshold_m. Returns the global default
    for empresas not in EMPRESA_CONFIG (graceful missing-key handling).

    Used by compute_pairs to decide which (front, back) pairs are cross-street
    contamination and should be filtered out.
    """
    cfg = EMPRESA_CONFIG.get(empresaid)
    if cfg is not None and cfg.lateral_pair_threshold_m_override is not None:
        return cfg.lateral_pair_threshold_m_override
    return PRODUCTIVE_PARAMS.lateral_pair_threshold_m

## Module: preprocessing/corridor

Construcción del trazado del corredor via PCA + binned median.

In [ ]:
"""Corridor centerline construction for the preprocessing pipeline.

Extracts an ordered (n_bins, 2) lat/lon polyline from GPS pings via:
  1. Geographic-outlier filter (IQR box trim at configurable quantiles).
  2. PCA to find the principal axis of the corridor.
  3. Binned median along the principal axis.
  4. Smoothing of the secondary (cross-corridor) coordinate.
  5. Back-transformation to (lat, lon).

Source: derived from build_notebook_03.py lines 279-361.
"""
from __future__ import annotations

import logging
import numpy as np
import polars as pl


logger = logging.getLogger(__name__)


def _filter_geographic_outliers(
    points_latlon: np.ndarray,
    q: tuple[float, float] = (
        PRODUCTIVE_PARAMS.centerline_latlon_quantile_lo,
        PRODUCTIVE_PARAMS.centerline_latlon_quantile_hi,
    ),
) -> np.ndarray:
    """Trim pings outside the [q_lo, q_hi] quantile box of lat and lon.

    Failure mode: if this filter is broken (too loose or too tight) the PCA
    principal axis tilts off-corridor or the sample becomes too small. The
    test_corridor.py outlier test catches regressions in both directions.

    Args:
        points_latlon: (n, 2) array of (lat, lon) values.
        q: (q_lo, q_hi) quantile tuple, default from PRODUCTIVE_PARAMS.

    Returns:
        Filtered (n_kept, 2) array.
    """
    pts = np.asarray(points_latlon, dtype=float)
    lat_lo, lat_hi = np.quantile(pts[:, 0], q)
    lon_lo, lon_hi = np.quantile(pts[:, 1], q)
    mask = (
        (pts[:, 0] >= lat_lo) & (pts[:, 0] <= lat_hi)
        & (pts[:, 1] >= lon_lo) & (pts[:, 1] <= lon_hi)
    )
    return pts[mask]


def build_centerline(
    gps: pl.DataFrame,
    empresaid: int,
    rng_seed: int = 42,
) -> np.ndarray:
    """Build the ordered (m, 2) lat/lon polyline for one empresa.

    Pipeline: geographic-outlier filter → PCA → binned median → trim → smooth
    → back-transform to (lat, lon).

    Args:
        gps: DataFrame with columns (empresaid, unidadid, lat, lon, speed_kmh).
             speed_kmh must already be populated — call
             projection.attach_observed_speed first.
        empresaid: which empresa to build the centerline for.
        rng_seed: seed for deterministic random sampling when the GPS sample
                  exceeds centerline_sample_cap.

    Returns:
        np.ndarray shape (m, 2) of (lat, lon) ordered along the principal axis,
        where m <= PRODUCTIVE_PARAMS.centerline_n_bins (bins with < 5 samples
        are silently dropped).

    Failure mode: PCA sign flip (centered data → eigenvector pointing west)
    produces a reversed polyline. test_corridor.py checks that the first vertex
    is near LON_START and the last is near LON_END of the synthetic route.
    """
    cfg = EMPRESA_CONFIG[empresaid]
    params = PRODUCTIVE_PARAMS

    moving = (
        gps.filter(
            (pl.col("empresaid") == empresaid)
            & (pl.col("speed_kmh") >= params.min_speed_for_centerline_kmh)
        )
        .select(["lat", "lon"])
    )

    rng = np.random.default_rng(rng_seed)
    sample: np.ndarray = moving.to_numpy()
    if len(sample) > cfg.centerline_sample_cap:
        idx = rng.choice(len(sample), size=cfg.centerline_sample_cap, replace=False)
        sample = sample[idx]

    return _build_centerline_from_points(
        sample,
        n_bins=params.centerline_n_bins,
        trim_pct=params.centerline_trim_pct,
        smooth_win=params.centerline_smooth_win,
    )


def build_centerline_per_direction(
    gps: pl.DataFrame,
    *,
    empresaid: int,
    direction_col: str = "direction",
    min_pings_per_dir: int = 1_000,
    rng_seed: int = 42,
) -> dict[int, np.ndarray]:
    """Build one (m, 2) centerline per direction key {+1, -1}.

    Filters gps by empresaid and speed >= min_speed_for_centerline_kmh, partitions
    by direction_col, calls _build_centerline_from_points per subset. Subsets below
    min_pings_per_dir fall back to single-pass build_centerline; same fallback on
    ValueError from sparse bins. Logs a structured FallbackEvent per fallback.

    Args:
        gps: DataFrame with columns (empresaid, direction, speed_kmh, lat, lon).
             speed_kmh must already be populated.
        empresaid: which empresa to build centerlines for.
        direction_col: name of the direction column (default "direction").
        min_pings_per_dir: minimum pings required per direction subset to attempt
                           per-direction PCA. Below this, falls back to single-pass
                           centerline. (R-CL1)
        rng_seed: seed for deterministic random sampling in the fallback single-pass
                  build_centerline call.

    Returns:
        dict[int, np.ndarray] with keys +1 and -1. Each value is the (m, 2)
        centerline for that direction subset. When a subset falls back to the
        single-pass centerline, that centerline is stored for the direction key.

    Raises:
        Never raises — all exceptions from _build_centerline_from_points trigger
        the fallback path.
    """
    params = PRODUCTIVE_PARAMS

    # Filter to this empresa's moving pings.
    # Speed filter is applied only when speed_kmh column is present
    # (it may be absent in test fixtures that pre-set direction without going
    # through attach_observed_speed).
    if "speed_kmh" in gps.columns:
        moving = gps.filter(
            (pl.col("empresaid") == empresaid)
            & (pl.col("speed_kmh") >= params.min_speed_for_centerline_kmh)
        )
    else:
        moving = gps.filter(pl.col("empresaid") == empresaid)

    # Build the single-pass centerline once (used as fallback for sparse directions)
    single_pass_cl = build_centerline(gps, empresaid=empresaid, rng_seed=rng_seed)

    result: dict[int, np.ndarray] = {}

    for direction in [1, -1]:
        subset = moving.filter(pl.col(direction_col) == direction)
        n_pings = subset.height

        if n_pings < min_pings_per_dir:
            logger.warning(
                "FallbackEvent: empresaid=%d direction=%d pings=%d reason=sparse "
                "(below min_pings_per_dir=%d); using single-pass centerline",
                empresaid, direction, n_pings, min_pings_per_dir,
            )
            result[direction] = single_pass_cl
            continue

        points = subset.select(["lat", "lon"]).to_numpy()
        try:
            cl = _build_centerline_from_points(
                points,
                n_bins=params.centerline_n_bins,
                trim_pct=params.centerline_trim_pct,
                smooth_win=params.centerline_smooth_win,
            )
            result[direction] = cl
        except ValueError as exc:
            logger.warning(
                "FallbackEvent: empresaid=%d direction=%d pings=%d reason=pca_error "
                "(%s); using single-pass centerline",
                empresaid, direction, n_pings, exc,
            )
            result[direction] = single_pass_cl

    return result


def _build_centerline_from_points(
    points_latlon: np.ndarray,
    n_bins: int = PRODUCTIVE_PARAMS.centerline_n_bins,
    trim_pct: float = PRODUCTIVE_PARAMS.centerline_trim_pct,
    smooth_win: int = PRODUCTIVE_PARAMS.centerline_smooth_win,
) -> np.ndarray:
    """Inner implementation of centerline construction from a point array.

    Separated from build_centerline to make the algorithm unit-testable with
    arbitrary point sets (not tied to a polars DataFrame or empresa).

    Args:
        points_latlon: (n, 2) array of (lat, lon) values.
        n_bins: number of bins along the principal axis.
        trim_pct: fraction of extreme principal-axis positions to drop.
        smooth_win: rolling mean window for the cross-corridor coordinate.

    Returns:
        np.ndarray shape (m, 2) of (lat, lon), m <= n_bins.
    """
    pts = _filter_geographic_outliers(points_latlon)

    centroid = pts.mean(axis=0)
    centered = pts - centroid

    # PCA via eigen-decomposition of the 2×2 covariance matrix.
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, order]

    projected = centered @ eigvecs     # (n, 2)
    t1 = projected[:, 0]               # principal axis coordinate
    t2 = projected[:, 1]               # cross-corridor coordinate

    # Trim extreme percentiles along the principal axis.
    lo, hi = np.quantile(t1, [trim_pct, 1.0 - trim_pct])
    mask = (t1 >= lo) & (t1 <= hi)
    t1, t2 = t1[mask], t2[mask]

    # Bin along the principal axis; take median cross-corridor coord per bin.
    bins = np.linspace(t1.min(), t1.max(), n_bins + 1)
    bin_idx = np.clip(np.digitize(t1, bins) - 1, 0, n_bins - 1)

    cl_proj: list[list[float]] = []
    for i in range(n_bins):
        m = bin_idx == i
        if m.sum() < 5:
            continue
        cl_proj.append([0.5 * (bins[i] + bins[i + 1]), float(np.median(t2[m]))])

    if not cl_proj:
        raise ValueError(
            f"build_centerline produced no bins with >= 5 points for n_bins={n_bins}. "
            "The GPS sample may be too small or too sparse."
        )

    cl_proj_arr = np.array(cl_proj)

    # Smooth the cross-corridor coordinate with a rolling mean.
    if smooth_win > 1 and len(cl_proj_arr) >= smooth_win:
        kernel = np.ones(smooth_win) / smooth_win
        cl_proj_arr[:, 1] = np.convolve(cl_proj_arr[:, 1], kernel, mode="same")

    # Back-transform from PCA space to (lat, lon).
    cl_latlon: np.ndarray = cl_proj_arr @ eigvecs.T + centroid

    # Enforce forward orientation along dominant PCA axis (eigvec sign is non-deterministic).
    # numpy.linalg.eigh may return a principal eigenvector whose dominant component is
    # negative (e.g., pointing south for a north-south route). When that happens, the
    # back-transformed polyline runs from the 'high-lat' end to the 'low-lat' end, causing
    # arc-length s to decrease as buses travel forward — which breaks infer_direction and
    # the trajectory matching in compute_headways_c2. Fix: if the dominant component of
    # eigvecs[:, 0] is negative, reverse the polyline so it always traverses in the
    # direction of the positive dominant geographic axis.
    dominant_idx = int(np.argmax(np.abs(eigvecs[:, 0])))
    if eigvecs[dominant_idx, 0] < 0:
        cl_latlon = cl_latlon[::-1]

    return cl_latlon

## Module: preprocessing/projection

Speed observado (`step_m / dt_s`) y proyección arc-length `s`.
Filtra pings off-route con `lateral_m > LATERAL_OFFSET_THRESHOLD_M`.

In [ ]:
"""Speed attachment and arc-length projection for the preprocessing pipeline.

Provides:
  attach_observed_speed — compute step_m, dt_s, speed_kmh per (empresaid, unidadid)
                          and DROP GPS-jump pairs per spec R11 (pair-level discard,
                          not row-level nulling).
  project_to_centerline — project pings onto a polyline, compute s and lateral_m,
                          drop off-route rows.

Source: derived from build_notebook_03.py lines 246-273 (speed) and
        390-468 (projection + off-route filter).
"""
from __future__ import annotations

import numpy as np
import polars as pl



def attach_observed_speed(gps: pl.DataFrame) -> pl.DataFrame:
    """Add columns (lat_prev, lon_prev, time_prev, step_m, dt_s, speed_kmh) by
    diffing successive rows of the same (empresaid, unidadid), then discard
    GPS-jump pairs per spec R11.

    speed_kmh is computed as step_m / dt_s * 3.6 (observed speed from GPS
    displacement). The raw `velocidad` field is intentionally NOT used (spec R11,
    decisiones-limpieza-fase2 §2.3).

    Pair-level discard (spec R11) — rows are DROPPED (not nulled) when:
      1. speed_kmh > MAX_PLAUSIBLE_SPEED_KMH (80 km/h): GPS jump or data error.
      2. step_m > MAX_PLAUSIBLE_JUMP_M (500 m) AND dt_s <= 60 s: implausible jump.

    The first ping per bus has no previous ping, so step_m and dt_s are null
    and speed_kmh is null. These rows are KEPT (null speed is not an outlier —
    it is missing data for the leading ping only). The filter conditions
    explicitly preserve null-speed rows.

    Output frame has fewer rows than input when GPS jumps are present.

    Source: build_notebook_03.py lines 250-272 (extended for R11 pair-level discard).
    """
    gps = gps.with_columns([
        pl.col("lat").shift(1).over(["empresaid", "unidadid"]).alias("lat_prev"),
        pl.col("lon").shift(1).over(["empresaid", "unidadid"]).alias("lon_prev"),
        pl.col("time").shift(1).over(["empresaid", "unidadid"]).alias("time_prev"),
    ])
    gps = gps.with_columns([
        (
            ((pl.col("lat") - pl.col("lat_prev")) * LAT_DEG_M) ** 2
            + ((pl.col("lon") - pl.col("lon_prev")) * LON_DEG_M) ** 2
        ).sqrt().alias("step_m"),
        (pl.col("time") - pl.col("time_prev")).dt.total_seconds().alias("dt_s"),
    ])
    gps = gps.with_columns(
        pl.when(pl.col("dt_s").is_not_null() & (pl.col("dt_s") > 0))
          .then(pl.col("step_m") / pl.col("dt_s") * 3.6)
          .otherwise(None)
          .alias("speed_kmh")
    )
    # Pair-level discard criterion 1 (spec R11): drop rows where speed > 80 km/h.
    # Null speed (first ping per bus) is preserved — it is not a GPS-jump outlier.
    gps = gps.filter(
        pl.col("speed_kmh").is_null() | (pl.col("speed_kmh") <= MAX_PLAUSIBLE_SPEED_KMH)
    )
    # Pair-level discard criterion 2 (spec R11): drop rows where step_m > 500 m
    # AND dt_s <= 60 s. This catches teleporting pings that briefly exceed the
    # jump threshold within a 1-minute window.
    # The first ping per bus has step_m = null (no previous ping) — these must
    # be kept. Polars propagates null through comparisons, so we must explicitly
    # preserve null-step_m rows with step_m.is_null() as an OR guard.
    gps = gps.filter(
        pl.col("step_m").is_null()
        | ~(
            (pl.col("step_m") > MAX_PLAUSIBLE_JUMP_M)
            & (pl.col("dt_s") <= 60)
        )
    )
    return gps


def project_to_centerline(
    gps: pl.DataFrame,
    centerline_latlon: np.ndarray,
    empresaid: int,
    chunk_size: int = 10_000,
) -> pl.DataFrame:
    """Project each ping onto the centerline polyline, compute arc-length s and
    lateral_m, then drop pings where lateral_m > lateral_threshold_for(empresaid).

    Args:
        gps: rows belonging to a SINGLE empresa with columns
             (empresaid, unidadid, time, lat, lon, speed_kmh).
        centerline_latlon: (m, 2) array of (lat, lon) from corridor.build_centerline.
        empresaid: used to look up the lateral offset threshold.
        chunk_size: number of pings to process per numpy batch (bounds peak memory).

    Returns:
        pl.DataFrame with added columns (s: Float64, lateral_m: Float64) after
        applying the lateral off-route filter. Pings with lateral_m above the
        threshold are removed.

    Failure mode: if chunk boundaries produce s discontinuities, monotonicity
    of s for a straight on-route bus breaks. test_projection.py catches this.

    Source: build_notebook_03.py lines 390-468.
    """
    gps_e = gps.filter(pl.col("empresaid") == empresaid)
    if gps_e.is_empty():
        return gps_e.with_columns([
            pl.lit(None, dtype=pl.Float64).alias("s"),
            pl.lit(None, dtype=pl.Float64).alias("lateral_m"),
        ])

    points_latlon = gps_e.select(["lat", "lon"]).to_numpy()
    s_arr, lateral_arr = _project_arc_length(points_latlon, centerline_latlon, chunk_size)

    threshold = lateral_threshold_for(empresaid)
    result = gps_e.with_columns([
        pl.Series("s", s_arr.astype(float), dtype=pl.Float64),
        pl.Series("lateral_m", lateral_arr.astype(float), dtype=pl.Float64),
    ]).filter(pl.col("lateral_m") <= threshold)

    return result


def project_per_direction(
    gps: pl.DataFrame,
    centerlines: dict[int, "np.ndarray"],
    *,
    empresaid: int,
    direction_col: str = "direction",
    chunk_size: int = 10_000,
) -> pl.DataFrame:
    """Project each direction subset onto its own centerline, then vertical_concat.

    Pings with direction not in centerlines (e.g. direction == 0) get NaN s,
    NaN lateral_m, and are kept (downstream filters handle them). Schema and dtypes
    match project_to_centerline.

    This function OVERWRITES the s and lateral_m columns in the returned DataFrame.
    It is designed for pass-2 of the two-pass pipeline: after pass-1 has already
    written s/lateral_m, call this to replace them with per-direction projections.

    Args:
        gps: DataFrame with columns including (direction, lat, lon) and existing
             s/lateral_m columns (will be overwritten). All rows are kept.
        centerlines: dict mapping direction int → (m, 2) centerline array.
                     Keys are typically {+1, -1}. Pings with unknown direction
                     keys receive NaN s and NaN lateral_m.
        empresaid: empresa identifier (for type consistency; not used for filtering
                   since gps is assumed to be already empresa-filtered).
        direction_col: name of the direction column (default "direction").
        chunk_size: number of pings per numpy batch (bounds peak memory).

    Returns:
        pl.DataFrame with same schema as input, same row count, with s and
        lateral_m overwritten by per-direction projections.
    """
    known_directions = set(centerlines.keys())
    parts: list[pl.DataFrame] = []

    for direction, cl in centerlines.items():
        subset = gps.filter(pl.col(direction_col) == direction)
        if subset.is_empty():
            continue
        pts = subset.select(["lat", "lon"]).to_numpy()
        s_arr, lateral_arr = _project_arc_length(pts, cl, chunk_size)
        subset = subset.with_columns([
            pl.Series("s", s_arr.astype(float), dtype=pl.Float64),
            pl.Series("lateral_m", lateral_arr.astype(float), dtype=pl.Float64),
        ])
        parts.append(subset)

    # Handle pings with unknown direction (not in centerlines) — assign NaN
    unknown_mask = ~pl.col(direction_col).is_in(list(known_directions))
    unknown_subset = gps.filter(unknown_mask)
    if not unknown_subset.is_empty():
        unknown_subset = unknown_subset.with_columns([
            pl.lit(float("nan"), dtype=pl.Float64).alias("s"),
            pl.lit(float("nan"), dtype=pl.Float64).alias("lateral_m"),
        ])
        parts.append(unknown_subset)

    if not parts:
        # Edge case: empty input
        return gps.with_columns([
            pl.lit(float("nan"), dtype=pl.Float64).alias("s"),
            pl.lit(float("nan"), dtype=pl.Float64).alias("lateral_m"),
        ])

    return pl.concat(parts)


def _project_arc_length(
    points_latlon: np.ndarray,
    centerline_latlon: np.ndarray,
    chunk_size: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Pure-numpy point-to-polyline projection using local flat-Earth coordinates.

    For each point, finds the closest centerline segment, projects orthogonally,
    and computes cumulative arc-length s (meters from polyline start) plus
    lateral offset (perpendicular distance in meters).

    Memory: O(chunk_size × n_segments) intermediate tensor. chunk_size=10_000
    with 50 segments ≈ 4 MB float32 — bounded regardless of total ping count.

    Source: build_notebook_03.py lines 396-427.
    """
    pts = np.asarray(points_latlon, dtype=float)
    cl = np.asarray(centerline_latlon, dtype=float)

    # Convert to meters (local flat-Earth at Arequipa latitude).
    pts_m = np.stack([pts[:, 0] * LAT_DEG_M, pts[:, 1] * LON_DEG_M], axis=1)
    cl_m = np.stack([cl[:, 0] * LAT_DEG_M, cl[:, 1] * LON_DEG_M], axis=1)

    seg_starts = cl_m[:-1]                              # (m-1, 2)
    seg_vecs = np.diff(cl_m, axis=0)                    # (m-1, 2)
    seg_norms_sq = (seg_vecs ** 2).sum(axis=1)          # (m-1,)
    seg_lengths = np.sqrt(seg_norms_sq)
    cum_s = np.concatenate([[0.0], np.cumsum(seg_lengths)])   # (m,)

    n = pts_m.shape[0]
    s_out = np.zeros(n, dtype=np.float32)
    lateral_out = np.zeros(n, dtype=np.float32)

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        chunk = pts_m[start:end]                                       # (c, 2)
        diff = chunk[:, None, :] - seg_starts[None, :, :]             # (c, m-1, 2)
        t = (diff * seg_vecs[None, :, :]).sum(axis=2) / seg_norms_sq[None, :]
        t = np.clip(t, 0.0, 1.0)                                       # (c, m-1)
        proj = seg_starts[None, :, :] + t[:, :, None] * seg_vecs[None, :, :]
        dist_sq = ((chunk[:, None, :] - proj) ** 2).sum(axis=2)       # (c, m-1)
        best_seg = dist_sq.argmin(axis=1)                              # (c,)
        best_t = np.take_along_axis(t, best_seg[:, None], axis=1).squeeze(1)
        s_out[start:end] = cum_s[best_seg] + best_t * seg_lengths[best_seg]
        lateral_out[start:end] = np.sqrt(
            np.take_along_axis(dist_sq, best_seg[:, None], axis=1).squeeze(1)
        )

    return s_out, lateral_out

## Module: preprocessing/direction

Inferencia de sentido ida/vuelta desde `sign(rolling_mean(ds, win=5))`.
E4 reporta `direccion` (has_heading=True), usado como verificación cruzada.

In [ ]:
"""Direction inference from the sign of the smoothed arc-length derivative.

Primary method: sign(rolling_mean(ds, DIRECTION_SMOOTH_WIN)) per (empresaid, unidadid).
This is the SOLE primary source (decisiones-limpieza-fase2 §3.1). The `direccion`
heading field is used only as a cross-check diagnostic for empresas that have it
(E2/E4) and is never used to overwrite the primary signal.

Source: derived from build_notebook_03.py lines 471-504.
"""
from __future__ import annotations

import polars as pl



def infer_direction(gps: pl.DataFrame) -> pl.DataFrame:
    """Infer per-ping direction from sign(rolling_mean(ds, DIRECTION_SMOOTH_WIN)).

    Direction values:
      +1  = ida  (increasing s)
      -1  = vuelta (decreasing s)
       0  = undetermined (insufficient or ambiguous data at window start/end)

    Args:
        gps: must have (empresaid, unidadid, s) columns and be sorted by
             (empresaid, unidadid, time).

    Returns:
        gps + columns (ds_raw: Float64, ds_smooth: Float64, direction: Int8).

    Source: build_notebook_03.py lines 475-487.
    """
    win = PRODUCTIVE_PARAMS.direction_smooth_win

    gps = gps.with_columns([
        (
            pl.col("s") - pl.col("s").shift(1).over(["empresaid", "unidadid"])
        ).alias("ds_raw"),
    ])
    gps = gps.with_columns([
        pl.col("ds_raw")
          .rolling_mean(window_size=win, min_samples=1)
          .over(["empresaid", "unidadid"])
          .alias("ds_smooth"),
    ])
    gps = gps.with_columns([
        pl.when(pl.col("ds_smooth") > 0).then(pl.lit(1, dtype=pl.Int8))
          .when(pl.col("ds_smooth") < 0).then(pl.lit(-1, dtype=pl.Int8))
          .otherwise(pl.lit(0, dtype=pl.Int8))
          .alias("direction"),
    ])
    return gps


def cross_check_heading(gps: pl.DataFrame, empresaid: int) -> pl.DataFrame:
    """DIAGNOSTIC ONLY — add a heading_agrees column for empresas with GPS heading.

    For empresas with EMPRESA_CONFIG[e].has_heading = True, computes agreement
    between the primary direction signal and a threshold-based heading
    classification. Does NOT alter the primary `direction` column.

    - `direccion == 0` is treated as NULL (sentinel value, not north heading).
    - heading classified: 45–135° → +1 (ida), 225–315° → -1 (vuelta), else 0.
    - heading_agrees = (primary direction == heading direction) when both non-zero.

    For empresas without heading (has_heading=False, e.g. E59) this is a no-op
    that returns gps unchanged.

    Args:
        gps: frame with (empresaid, direction) columns.
        empresaid: empresa identifier.

    Returns:
        gps + column (heading_agrees: Boolean) when applicable; unchanged otherwise.
    """
    cfg = EMPRESA_CONFIG.get(empresaid)
    if cfg is None or not cfg.has_heading:
        return gps
    if "direccion" not in gps.columns:
        return gps

    gps = gps.with_columns([
        # Treat direccion == 0 as null (sentinel).
        pl.when(pl.col("direccion") == 0)
          .then(None)
          .otherwise(pl.col("direccion"))
          .alias("_heading_clean"),
    ])
    gps = gps.with_columns([
        pl.when(
            (pl.col("_heading_clean") >= 45) & (pl.col("_heading_clean") <= 135)
        ).then(pl.lit(1, dtype=pl.Int8))
          .when(
            (pl.col("_heading_clean") >= 225) & (pl.col("_heading_clean") <= 315)
        ).then(pl.lit(-1, dtype=pl.Int8))
          .otherwise(pl.lit(0, dtype=pl.Int8))
          .alias("_heading_dir"),
    ])
    gps = gps.with_columns([
        pl.when(
            (pl.col("direction") != 0) & (pl.col("_heading_dir") != 0)
        ).then(pl.col("direction") == pl.col("_heading_dir"))
          .otherwise(None)
          .alias("heading_agrees"),
    ]).drop(["_heading_clean", "_heading_dir"])

    return gps

## Module: preprocessing/trips

Segmentación de viajes (gap 30 min / reversal / terminal dwell 5 min)
y grilla de snapshots con alineación minuto-exacta (INV-6).

In [ ]:
"""Trip segmentation and snapshot grid construction.

assign_trip_ids — split each bus trajectory into trips on three cut conditions:
  1. GAP cut: dt_s > GAP_CUT_SECONDS (30 min) between consecutive pings.
  2. DIRECTION REVERSAL cut: primary direction flips +1↔-1 (transient 0s skipped).
  3. TERMINAL cut: bus stops near s_min/s_max for >= TERMINAL_DWELL_SECONDS (5 min).

build_snapshots — resample each bus to a minute-aligned uniform time grid, with
  linear interpolation of s and speed_kmh, and nearest-known direction by
  left-search.

Source: derived from build_notebook_03.py lines 606-660 (build_snapshots).
Trip segmentation is net-new for production — the probe deferred it.
"""
from __future__ import annotations

import numpy as np
import polars as pl



def _compute_trip_ids_for_bus(
    s_arr: np.ndarray,
    dt_s_arr: np.ndarray,
    speed_arr: np.ndarray,
    dir_arr: np.ndarray,
    time_arr: np.ndarray,
    s_min: float,
    s_max: float,
) -> np.ndarray:
    """Compute trip_id per ping for a single (empresaid, unidadid).

    Returns a uint32 array of the same length as the input arrays, where each
    element is the trip_id for that ping (monotonically non-decreasing).

    Cut conditions:
      1. GAP: dt_s > GAP_CUT_SECONDS
      2. REVERSAL: last-known direction flips +1 ↔ -1 (direction==0 skipped)
      3. TERMINAL EXIT: bus leaves a near-terminal stopped zone that lasted >= DWELL_SECONDS

    The terminal cut is placed on the EXIT ping (the first ping after leaving
    the dwell zone that exceeded the duration threshold).
    """
    n = len(s_arr)
    trip_ids = np.zeros(n, dtype=np.uint32)
    current_trip = np.uint32(0)

    # --- Pre-compute per-ping flags ---
    is_gap = np.zeros(n, dtype=bool)
    is_gap[1:] = dt_s_arr[1:] > GAP_CUT_SECONDS

    # REVERSAL: forward-fill direction ignoring 0s; detect sign flip.
    last_dir = 0
    prev_last_dir = 0
    is_reversal = np.zeros(n, dtype=bool)
    for i in range(n):
        d = int(dir_arr[i])
        if d != 0:
            if prev_last_dir != 0 and d != last_dir:
                # But we only cut on the non-zero flip, not on the first transition.
                # We set is_reversal at position i (the new direction starts here).
                is_reversal[i] = True
            prev_last_dir = last_dir
            last_dir = d

    # TERMINAL DWELL: track cumulative time near terminal while stopped.
    near = (s_arr < (s_min + TERMINAL_BAND_M)) | (s_arr > (s_max - TERMINAL_BAND_M))
    stopped = speed_arr < TERMINAL_MAX_SPEED_KMH
    in_dwell = near & stopped

    is_terminal_exit = np.zeros(n, dtype=bool)
    dwell_start_time = None
    dwell_block_exceeded = False

    for i in range(n):
        if in_dwell[i]:
            if dwell_start_time is None:
                dwell_start_time = time_arr[i]
                dwell_block_exceeded = False
            elapsed = float(time_arr[i] - dwell_start_time) / 1e9  # ns → s
            if elapsed >= TERMINAL_DWELL_SECONDS:
                dwell_block_exceeded = True
        else:
            if dwell_block_exceeded:
                # This is the EXIT ping.
                is_terminal_exit[i] = True
            dwell_start_time = None
            dwell_block_exceeded = False

    # --- Assemble trip_ids from cut flags ---
    for i in range(n):
        if i > 0 and (is_gap[i] or is_reversal[i] or is_terminal_exit[i]):
            current_trip += np.uint32(1)
        trip_ids[i] = current_trip

    return trip_ids


def assign_trip_ids(
    gps: pl.DataFrame,
    s_min: float | None = None,
    s_max: float | None = None,
) -> pl.DataFrame:
    """Assign a monotonic trip_id per (empresaid, unidadid) from three cut criteria.

    Cut conditions (any one triggers a new trip_id):
      1. GAP: dt_s > GAP_CUT_SECONDS between consecutive pings.
      2. REVERSAL: last-known direction flips +1 ↔ -1 (transient direction==0
         pings are skipped using forward-fill of the last non-zero direction).
      3. TERMINAL: bus is within TERMINAL_BAND_M of s_min or s_max AND stopped
         (speed_kmh < TERMINAL_MAX_SPEED_KMH) for >= TERMINAL_DWELL_SECONDS.
         The cut is placed on the EXIT ping of the dwell run (i.e. the first
         ping where the bus resumes movement or leaves the terminal band).

    trip_id is UInt32, monotonically increasing per bus, starting from 0 at
    the first ping of each (empresaid, unidadid). Trips of length < 2 pings
    are KEPT (downstream filters may drop them; we do not silently merge).

    Args:
        gps: must have (empresaid, unidadid, time, s, speed_kmh, direction, dt_s)
             sorted by (empresaid, unidadid, time).
        s_min: corridor start arc-length (meters). Computed from data if None.
        s_max: corridor end arc-length (meters). Computed from data if None.

    Returns:
        gps + column (trip_id: UInt32).

    Failure modes:
    - If terminal-cut boundary semantics flip (cut on ENTRY instead of EXIT),
      test_trips.py::test_terminal_cut_creates_new_trip_on_e59 catches it.
    - If reversal cut is placed on the direction==0 pings (short stops),
      trip count inflates; test_trips.py::test_gap_cut_creates_new_trip
      provides a stable baseline count.
    """
    if s_min is None:
        s_min = float(gps["s"].min() or 0.0)
    if s_max is None:
        s_max = float(gps["s"].max() or 0.0)

    gps = gps.sort(["empresaid", "unidadid", "time"])

    # Use row_index to guarantee correct positional mapping back to the sorted frame
    # after group_by (maintain_order=True guarantees group iteration order but not
    # row order within the full frame after re-join).
    gps_indexed = gps.with_row_index("_row_idx")
    trip_id_parts: list[pl.DataFrame] = []

    for keys, sub in gps_indexed.group_by(["empresaid", "unidadid"], maintain_order=True):
        sub_sorted = sub.sort("time")
        dt_s = sub_sorted["dt_s"].fill_null(0.0).to_numpy().astype(np.float64)
        s_arr = sub_sorted["s"].to_numpy().astype(np.float64)
        speed_arr = sub_sorted["speed_kmh"].fill_null(0.0).to_numpy().astype(np.float64)
        dir_arr = sub_sorted["direction"].to_numpy().astype(np.int64)
        time_arr = sub_sorted["time"].to_numpy().astype("datetime64[ns]").astype(np.int64)

        trip_ids = _compute_trip_ids_for_bus(
            s_arr, dt_s, speed_arr, dir_arr, time_arr, s_min, s_max
        )
        trip_id_parts.append(pl.DataFrame({
            "_row_idx": sub_sorted["_row_idx"],
            "trip_id": trip_ids,
        }))

    if not trip_id_parts:
        return gps.with_columns(pl.lit(0, dtype=pl.UInt32).alias("trip_id"))

    trip_id_df = pl.concat(trip_id_parts)
    result = gps_indexed.join(trip_id_df, on="_row_idx", how="left").drop("_row_idx")
    return result


def build_snapshots(
    gps: pl.DataFrame,
    grid_seconds: int = PRODUCTIVE_PARAMS.grid_seconds,
) -> pl.DataFrame:
    """Resample each bus to a minute-aligned uniform time grid per (empresaid, day).

    Grid alignment: uses epoch-floor pattern (t_min_s // grid_s) * grid_s to
    ensure all t_grid timestamps satisfy t.second == 0 (clarification #17 rule 1,
    INV-6).

    Interpolation:
      s         — linear interpolation (np.interp)
      speed_kmh — linear interpolation (null → 0.0 before interpolating)
      direction — nearest known by left-search (latest known state)
      trip_id   — nearest by left-search (when column is present)

    Only grid points within the bus's reported [t_min, t_max] window are emitted.
    Buses with < 2 pings are skipped.

    Source: build_notebook_03.py lines 606-660 with epoch-floor alignment added.
    """
    snaps_per_eday: list[pl.DataFrame] = []

    # Add day column if not present.
    if "day" not in gps.columns:
        gps = gps.with_columns(pl.col("time").dt.date().alias("day"))

    has_trip_id = "trip_id" in gps.columns
    has_lateral_m = "lateral_m" in gps.columns

    for keys, sub_eday in gps.group_by(["empresaid", "day"], maintain_order=True):
        e, day = keys[0], keys[1]

        # Compute minute-aligned epoch-floor grid (INV-6 / clarification #17 rule 1).
        # Use numpy int64 microseconds (matching polars Datetime["us"] storage) to
        # avoid Python datetime.timestamp() UTC/local ambiguity.
        t_min_us = int(sub_eday["time"].to_numpy().astype("datetime64[us]").astype(np.int64).min())
        t_max_us = int(sub_eday["time"].to_numpy().astype("datetime64[us]").astype(np.int64).max())
        grid_us = grid_seconds * 1_000_000   # grid in microseconds
        t_grid_us = np.arange(
            (t_min_us // grid_us) * grid_us,
            ((t_max_us // grid_us) + 1) * grid_us + 1,
            grid_us,
            dtype=np.int64,
        )
        # Also keep ns for interp (t_arr will be ns from the per-bus conversion below).
        t_grid_ns = t_grid_us * 1_000

        for bus_keys, sub in sub_eday.group_by(["unidadid"], maintain_order=True):
            bus = bus_keys[0]
            sub_sorted = sub.sort("time")
            t_arr = sub_sorted["time"].to_numpy().astype("datetime64[us]").astype(np.int64) * 1_000
            s_arr = sub_sorted["s"].to_numpy().astype(np.float64)
            v_arr = sub_sorted["speed_kmh"].fill_null(0.0).to_numpy().astype(np.float64)
            d_arr = sub_sorted["direction"].to_numpy().astype(np.int64)

            if len(t_arr) < 2:
                continue

            # Only interpolate within the bus's reported window.
            in_window = (t_grid_ns >= t_arr[0]) & (t_grid_ns <= t_arr[-1])
            if not in_window.any():
                continue

            tg = t_grid_ns[in_window]
            s_interp = np.interp(tg, t_arr, s_arr)
            v_interp = np.interp(tg, t_arr, v_arr)

            # Direction: nearest known (left-search), carrying the latest known state.
            idx_left = np.searchsorted(t_arr, tg, side="right") - 1
            idx_left = np.clip(idx_left, 0, len(d_arr) - 1)
            d_interp = d_arr[idx_left]

            row_data: dict = {
                "empresaid": np.full(len(tg), int(e), dtype=np.int64),
                "day": [day] * len(tg),
                "t": tg,
                "unidadid": np.full(len(tg), int(bus), dtype=np.int64),
                "s": s_interp.astype(np.float64),
                "speed_kmh": v_interp.astype(np.float64),
                "direction": d_interp.astype(np.int8),
            }

            if has_trip_id:
                tid_arr = sub_sorted["trip_id"].to_numpy().astype(np.uint32)
                idx_trip = np.searchsorted(t_arr, tg, side="right") - 1
                idx_trip = np.clip(idx_trip, 0, len(tid_arr) - 1)
                row_data["trip_id"] = tid_arr[idx_trip]

            if has_lateral_m:
                # Linear interpolation of lateral_m alongside s/speed_kmh.
                # lateral_m is a continuous geometric quantity (orthogonal distance
                # to centerline) — same regularity class as s. np.interp handles
                # null/NaN by propagating them; fill_null(0.0) is intentionally NOT
                # used here because a null lateral_m carries meaning (ping without
                # projection), and we want to propagate it faithfully.
                lat_arr = sub_sorted["lateral_m"].fill_null(float("nan")).to_numpy().astype(np.float64)
                lat_interp = np.interp(tg, t_arr, lat_arr)
                # Convert NaN back to null via a float64 series.
                lat_series = pl.Series("lateral_m", lat_interp, dtype=pl.Float64)
                row_data["lateral_m"] = lat_interp

            snaps_per_eday.append(pl.DataFrame(row_data))

    if not snaps_per_eday:
        return pl.DataFrame()

    snaps = pl.concat(snaps_per_eday)
    # t is stored as int64 nanoseconds (from t_grid_ns); convert to Datetime[us].
    snaps = snaps.with_columns(
        (pl.col("t") // 1_000).cast(pl.Datetime("us")).alias("t")
    )
    return snaps

## Module: preprocessing/headways

C.2 trailing crossing — pure polars+numpy. Para pares sin historial previo
se emite `delta_t_min = null` (NO se descarta). Winsorización en Fase 5.

In [ ]:
"""Headway computation via C.2 — trailing crossing (pure polars + numpy).

compute_pairs — build the pair structure: for each (empresaid, day, t, direction),
    sort buses by s and emit one (front, back) row per consecutive pair.

compute_headways_c2 — for each pair, find the most recent past time when bus_back
    crossed s_front in its trajectory, and compute delta_t_min = T - t_cross.

Clarification #17 rule 2: when no crossing is found, the row is EMITTED with
delta_t_min = null (NOT dropped). This preserves pair_rank density (INV-3) and
n_buses consistency.

Note on NULL rows: they appear mostly in the first GRID_SECONDS of a bus's
trajectory (before bus_back has driven through any front position). The NULL
fraction should be < 5% globally; if higher, investigate trip-segmentation edge
cases. (Caveat per clarification #17 §Frequency expectation.)

Source: rewrite of build_notebook_03.py lines 751-813. The probe used pandas
    conversion + row-level Python loop. This implementation uses a trajectory
    index built with polars group_by + numpy numpy-escape per back-bus group
    (O(K) per group, not per pair).

winsorization: delta_t_min is stored RAW. Winsorization is a Fase 5 transformation
    applied at training time, NOT here (decisiones-headway-fase2.md §4 Caveat 2).
"""
from __future__ import annotations

import logging
from collections import Counter

import numpy as np
import polars as pl


logger = logging.getLogger(__name__)


def compute_pairs(snapshots: pl.DataFrame) -> pl.DataFrame:
    """Build consecutive (front, back) bus pairs per (empresaid, day, t, direction).

    For each snapshot group sorted by s (ascending), bus at rank i is "front" and
    bus at rank i-1 is "back". Drops direction == 0 rows.

    Lateral pair filter (R-LAT3): after pair formation, drops pairs where
    |lateral_m_front − lateral_m_back| > lateral_pair_threshold_for(empresaid).
    Rows where either lateral value is null are RETAINED (conservative).
    Filter is applied only when the input snapshot frame contains a `lateral_m`
    column. When the column is absent, all pairs are retained (backward-compatible).

    Args:
        snapshots: output of trips.build_snapshots with columns
                   (empresaid, day, t, unidadid, s, speed_kmh, direction[, lateral_m]).

    Returns:
        pl.DataFrame with columns:
          empresaid, day, t, direction,
          pair_rank (Int32, 1-indexed, dense per group),
          bus_front (Int64), bus_back (Int64),
          s_front (Float64), s_back (Float64),
          speed_front_kmh (Float64), speed_back_kmh (Float64),
          n_buses (Int32)[, lateral_m_front (Float64), lateral_m_back (Float64)].
          The lateral columns are present only when the input has lateral_m.

    Failure mode: if shift(1) is applied before sort, pair assignment is wrong;
    test_headways.py::test_pair_structure_count catches this.
    """
    has_lateral = "lateral_m" in snapshots.columns

    s = snapshots.filter(pl.col("direction") != 0)
    # Direction-conditional sort key (SDD dir1-pair-ordering-h7, Encoding A).
    # For CALIBRATED_INVERTED_DIRECTION (+1): the per-direction PCA centerline
    # has s inverse to physical motion → negate s so ascending sort places the
    # physically-front bus last (it becomes bus_front after shift(1)).
    # For the canonical direction: sort key == s (identical to pre-fix behavior).
    # The negation is sort-time only; s_front/s_back retain raw arc-length values.
    _s_sort_key = (
        pl.when(pl.col("direction") == CALIBRATED_INVERTED_DIRECTION)
        .then(-pl.col("s"))
        .otherwise(pl.col("s"))
    )
    s = s.sort(["empresaid", "day", "t", "direction", _s_sort_key])

    group_cols = ["empresaid", "day", "t", "direction"]

    shift_exprs = [
        pl.col("s").shift(1).over(group_cols).alias("s_back"),
        pl.col("unidadid").shift(1).over(group_cols).alias("bus_back"),
        pl.col("speed_kmh").shift(1).over(group_cols).alias("speed_back_kmh"),
        pl.col("unidadid").count().over(group_cols).cast(pl.Int32).alias("n_buses"),
        # cum_count starts at 1 for the first row; after dropping the first row
        # (the "back" reference is null) we get ranks 2..N. Subtract 1 to get 1..N-1.
        (pl.col("s").cum_count().over(group_cols).cast(pl.Int32) - 1).alias("pair_rank"),
    ]
    if has_lateral:
        # Shift lateral_m to get the back-bus value after pairing.
        shift_exprs.append(
            pl.col("lateral_m").shift(1).over(group_cols).alias("lateral_m_back_raw")
        )
        # The front bus keeps its own lateral_m (renamed after select).
        shift_exprs.append(
            pl.col("lateral_m").alias("lateral_m_front_raw")
        )

    s = s.with_columns(shift_exprs)

    # Drop the first bus in each group (shift produces null for it).
    s = s.filter(pl.col("s_back").is_not_null())

    if has_lateral:
        # Step 6: filter cross-street pairs.
        # Build per-empresa threshold mapping via Python-side lookup (task note:
        # fallback from vectorised when/then if empresa list varies).
        empresa_ids = s["empresaid"].unique().to_list()
        threshold_map = {int(e): lateral_pair_threshold_for(int(e)) for e in empresa_ids}

        # Build a Polars expression: pl.col("empresaid").replace(mapping, default=global)
        # Conservative rule: retain if either lateral value is null.
        # retain when: lateral_m_front_raw IS NULL
        #           OR lateral_m_back_raw IS NULL
        #           OR abs(front - back) <= threshold
        global_threshold = PRODUCTIVE_PARAMS.lateral_pair_threshold_m
        keep_expr = (
            pl.col("lateral_m_front_raw").is_null()
            | pl.col("lateral_m_back_raw").is_null()
            | (
                (pl.col("lateral_m_front_raw") - pl.col("lateral_m_back_raw")).abs()
                <= pl.col("empresaid").replace_strict(
                    threshold_map,
                    default=global_threshold,
                    return_dtype=pl.Float64,
                )
            )
        )
        s = s.filter(keep_expr)

    select_exprs = [
        "empresaid",
        "day",
        "t",
        "direction",
        "pair_rank",
        pl.col("unidadid").alias("bus_front"),
        pl.col("bus_back").cast(pl.Int64),
        pl.col("s").alias("s_front"),
        pl.col("s_back").cast(pl.Float64),
        pl.col("speed_kmh").alias("speed_front_kmh"),
        pl.col("speed_back_kmh").cast(pl.Float64),
        "n_buses",
    ]
    if has_lateral:
        select_exprs += [
            pl.col("lateral_m_front_raw").cast(pl.Float64).alias("lateral_m_front"),
            pl.col("lateral_m_back_raw").cast(pl.Float64).alias("lateral_m_back"),
        ]

    return s.select(select_exprs)


# Canonical bucket names reported by _find_last_crossing_ns (5 paths).
# The 6th bucket ("traj-miss") is reported by the outer compute_headways_c2 loop.
_CROSSING_BUCKETS = ("success", "cutoff-lt-2", "no-crossing", "ds-zero", "stale-crossing")


def _find_last_crossing_ns(
    t_arr: np.ndarray,
    s_arr: np.ndarray,
    T_ns: int,
    s_front: float,
    max_lookback_ns: float | None = None,
) -> tuple[float | None, str]:
    """Find the most recent time (nanoseconds) when bus_back's s crossed s_front.

    Uses the probe's sign-change scan (build_notebook_03.py lines 796-806) on the
    trajectory of bus_back restricted to t <= T. Linear interpolation over the
    bracket gives the exact crossing nanosecond.

    Args:
        t_arr: int64 nanosecond timestamps, sorted ascending.
        s_arr: float64 arc-length values at those timestamps.
        T_ns:  snapshot time in nanoseconds (restrict to t <= T).
        s_front: arc-length of the front bus at T.
        max_lookback_ns: when not None, crossings older than this many nanoseconds
            before T are treated as 'no crossing found' and return None. Prevents
            stale historical crossings in multi-filar corridors (e.g. E2 Arequipa)
            from being emitted as absurd delta_t_min values (decisiones-headway-fase2 §3).

    Returns:
        (t_cross, bucket) where t_cross is in nanoseconds (float) or None,
        and bucket is one of _CROSSING_BUCKETS identifying the outcome.
    """
    cutoff = int(np.searchsorted(t_arr, T_ns, side="right"))
    if cutoff < 2:
        return None, "cutoff-lt-2"

    s_past = s_arr[:cutoff]
    t_past = t_arr[:cutoff]

    diff = s_past - s_front

    # Case 1: exact zero crossing — bus_back was exactly at s_front.
    zero_mask = diff == 0.0
    if zero_mask.any():
        i = int(np.where(zero_mask)[0][-1])
        t_cross = float(t_past[i])
        if max_lookback_ns is not None and (T_ns - t_cross) > max_lookback_ns:
            return None, "stale-crossing"
        return t_cross, "success"

    # Case 2: sign-change crossing — bus_back's s straddled s_front.
    signs = np.sign(diff)
    cross_mask = (signs[:-1] * signs[1:]) < 0

    if not cross_mask.any():
        return None, "no-crossing"

    # Most recent crossing (last True in cross_mask).
    i = int(np.where(cross_mask)[0][-1])

    ds = s_past[i + 1] - s_past[i]
    if ds == 0.0:
        return None, "ds-zero"

    frac = float((s_front - s_past[i]) / ds)
    t_cross = float(t_past[i]) + frac * float(t_past[i + 1] - t_past[i])
    if max_lookback_ns is not None and (T_ns - t_cross) > max_lookback_ns:
        return None, "stale-crossing"
    return t_cross, "success"


def compute_headways_c2(
    snapshots: pl.DataFrame,
    gps: pl.DataFrame,
    min_buses: int = PRODUCTIVE_PARAMS.min_buses_per_snapshot,
    max_lookback_minutes: float = PRODUCTIVE_PARAMS.max_interpolation_lookback_minutes,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """C.2 trailing-crossing headway (pure polars + numpy).

    For each pair (bus_front at s_front, bus_back) at snapshot time T, finds the
    most recent past time when bus_back's s-trajectory crossed s_front (in the
    same direction) and computes:

        delta_t_min = (T - t_cross).total_seconds() / 60

    When no crossing is found (e.g. bus_back just entered the corridor and has
    not yet crossed s_front): emits the row with delta_t_min = null (NOT dropped).
    This is clarification #17 rule 2 — preserves INV-3 (dense pair_rank) and
    INV-4 (n_buses consistent with active bus count).

    Crossings whose interpolated t_cross is older than max_lookback_minutes are
    treated as 'no crossing' and emitted with delta_t_min = NULL (same semantics
    as clarification §2). This bound exists because multi-filar corridors project
    unrelated buses to the same s axis; without it, np.searchsorted finds ancient
    crossings and emits absurd headways (e.g. ~112 days on E2 dir=1).

    Algorithm:
    1. Build a trajectory index: group gps by (empresaid, unidadid, direction)
       → (t_arr, s_arr) sorted by time. This is O(N) per group.
    2. Build the pair frame via compute_pairs.
    3. Iterate groups (empresaid, bus_back, direction): for all pairs in this
       group, run the numpy sign-change scan and record delta_t_min. This is
       O(P_k × K_k) per group where P_k = pairs for this back-bus and K_k = traj
       length. Reassemble via an explicit row-index join.

    Args:
        snapshots: output of trips.build_snapshots.
        gps: post-projection, post-direction frame (full trajectory for crossing
             lookup). Should be filtered to the relevant empresa and day range.
        min_buses: drop snapshot groups with fewer buses (INV-4).
        max_lookback_minutes: crossings older than this many minutes before T are
            emitted as NULL (same as no-crossing). Default from ProductiveParams.

    Returns:
        (headways_df, null_buckets_df) where:
          headways_df matches R7 schema:
            t, direction, pair_rank (Int32), bus_front (Int64), bus_back (Int64),
            s_front, s_back, speed_front_kmh, speed_back_kmh,
            delta_t_min (Float64, may be null per clarification #17 rule 2),
            n_buses (Int32).
          null_buckets_df schema (INV-N1 through INV-N4):
            empresaid (Int64), direction (Int8), bucket (Utf8),
            count (Int64), total_pairs (Int64).
            One row per (empresaid, direction, bucket) — always 6 buckets per group,
            count=0 rows included. INV-N2: sum(count) == total_pairs per group.

    Failure mode: if the pandas-conversion path is accidentally reintroduced,
    performance collapses on 47M-row E2 data. test_headways.py guards the polars
    purity requirement.
    """
    # Canonical bucket name ordering for null_buckets_df construction.
    _all_buckets = ("traj-miss", "cutoff-lt-2", "no-crossing", "ds-zero", "stale-crossing", "success")

    # Schema for null_buckets_df (locked; changes require spec revision).
    _null_buckets_schema = {
        "empresaid": pl.Int64,
        "direction": pl.Int8,
        "bucket": pl.Utf8,
        "count": pl.Int64,
        "total_pairs": pl.Int64,
    }

    pairs = compute_pairs(snapshots)

    # Drop pairs from too-small snapshots (INV-4: n_buses >= min_buses).
    pairs = pairs.filter(pl.col("n_buses") >= min_buses)
    if pairs.is_empty():
        empty_headways = pairs.with_columns(pl.lit(None, dtype=pl.Float64).alias("delta_t_min"))
        empty_null_buckets = pl.DataFrame(schema=_null_buckets_schema)
        return empty_headways, empty_null_buckets

    # Convert minutes → nanoseconds ONCE (kernel works in nanoseconds throughout).
    max_lookback_ns = float(max_lookback_minutes) * 60.0 * 1e9

    # --- Build trajectory index ---
    # Group gps by (empresaid, unidadid, direction) → sorted (t_arr, s_arr).
    gps_dir = gps.filter(pl.col("direction") != 0)
    traj_index: dict[tuple[int, int, int], tuple[np.ndarray, np.ndarray]] = {}

    for keys, sub in gps_dir.group_by(
        ["empresaid", "unidadid", "direction"], maintain_order=False
    ):
        e, bus, dirc = int(keys[0]), int(keys[1]), int(keys[2])
        # Use microsecond-based int64 (Datetime["us"]) × 1000 → nanoseconds.
        t_arr = sub["time"].to_numpy().astype("datetime64[us]").astype(np.int64) * 1_000
        s_arr = sub["s"].to_numpy().astype(np.float64)
        order = np.argsort(t_arr)
        traj_index[(e, bus, dirc)] = (t_arr[order], s_arr[order])

    # --- Compute delta_t_min per pair ---
    # Attach a row index to pairs for result reassembly.
    pairs_indexed = pairs.with_row_index("_row_idx")

    # t column: snapshots use Datetime["us"], convert to nanoseconds for the lookup.
    t_ns_all = (
        pairs_indexed["t"].to_numpy().astype("datetime64[us]").astype(np.int64) * 1_000
    )
    s_front_all = pairs_indexed["s_front"].to_numpy().astype(np.float64)

    n = len(pairs_indexed)
    delta_t_min = np.full(n, np.nan, dtype=np.float64)

    # Bucket counter: accumulate per (empresaid, direction, bucket) across all pairs.
    # CORRECTNESS NOTE: total_counter counts PAIRS (not groups). Each increment by
    # len(sub_idx) (the number of rows/pairs in the group) ensures the discrimination
    # invariant INV-N2: sum(count over all buckets) == total_pairs per (e, d).
    # The previous code used += 1 (counting groups, not pairs) — that was wrong.
    bucket_counter: Counter[tuple[int, int, str]] = Counter()
    total_counter: Counter[tuple[int, int]] = Counter()

    # Iterate per (empresaid, bus_back, direction) group — O(P_k × K_k) per group.
    for keys, sub_idx in pairs_indexed.group_by(
        ["empresaid", "bus_back", "direction"], maintain_order=False
    ):
        e, bus, dirc = int(keys[0]), int(keys[1]), int(keys[2])
        # Count pairs in this group (not the group itself — correctness fix).
        total_counter[(e, dirc)] += len(sub_idx)
        traj_key = (e, bus, dirc)
        if traj_key not in traj_index:
            # All pairs in this group are traj-miss.
            bucket_counter[(e, dirc, "traj-miss")] += len(sub_idx)
            continue

        t_arr, s_arr = traj_index[traj_key]

        row_indices = sub_idx["_row_idx"].to_numpy().astype(np.int64)
        T_ns_group = t_ns_all[row_indices]
        s_front_group = s_front_all[row_indices]

        for j, (T_ns, sf) in enumerate(zip(T_ns_group, s_front_group)):
            t_cross, reason = _find_last_crossing_ns(
                t_arr, s_arr, int(T_ns), float(sf),
                max_lookback_ns=max_lookback_ns,
            )
            # Count each pair outcome by its bucket (1 per pair call).
            bucket_counter[(e, dirc, reason)] += 1
            if t_cross is not None:
                dt_ns = float(T_ns) - t_cross
                delta_t_min[row_indices[j]] = dt_ns / 1e9 / 60.0

    # Reassemble: NaN → null (clarification #17 rule 2 — emit null NOT drop).
    delta_series = pl.Series("delta_t_min", delta_t_min, dtype=pl.Float64)
    delta_series = delta_series.set(delta_series.is_nan(), None)

    result = pairs_indexed.drop("_row_idx").with_columns(delta_series)

    # --- Build null_buckets_df from counters ---
    # All 6 buckets per (empresaid, direction) with count=0 when bucket didn't fire.
    # This satisfies INV-N2: sum(count) == total_pairs for every group.
    bucket_rows = []
    for (e, d), total in total_counter.items():
        for b in _all_buckets:
            bucket_rows.append({
                "empresaid": e,
                "direction": d,
                "bucket": b,
                "count": int(bucket_counter.get((e, d, b), 0)),
                "total_pairs": int(total),
            })
    null_buckets_df = pl.DataFrame(bucket_rows, schema=_null_buckets_schema)

    # Emit per-(empresa, direction) trajectory-miss diagnostics.
    # Prefix [traj-miss] is machine-grepable; [traj-miss-warning] fires when > 30%.
    for (e, d), total in total_counter.items():
        miss = bucket_counter.get((e, d, "traj-miss"), 0)
        pct = (miss / total * 100.0) if total else 0.0
        logger.info(
            "[traj-miss] empresa=%d dir=%d miss=%d/%d (%.1f%%)",
            e, d, miss, total, pct,
        )
        if pct > 30.0:
            logger.warning(
                "[traj-miss-warning] empresa=%d dir=%d miss_pct=%.1f%% exceeds 30%%",
                e, d, pct,
            )

    # Final schema cleanup: select R7 columns, preserving lateral diagnostic
    # columns when the upstream compute_pairs emitted them (R-LAT4 / AC-S1 / AC-S2).
    r7_cols = [
        "t",
        "direction",
        "pair_rank",
        "bus_front",
        "bus_back",
        "s_front",
        "s_back",
        "speed_front_kmh",
        "speed_back_kmh",
        "delta_t_min",
        "n_buses",
    ]
    if "lateral_m_front" in pairs_indexed.columns:
        r7_cols.append("lateral_m_front")
    if "lateral_m_back" in pairs_indexed.columns:
        r7_cols.append("lateral_m_back")
    return result.select(r7_cols), null_buckets_df


def filter_snapshot_size(headways: pl.DataFrame, min_buses: int) -> pl.DataFrame:
    """Drop rows belonging to snapshots with fewer than min_buses active buses.

    INV-4: n_buses >= min_buses for all rows.

    Implementation: filter on the pre-computed n_buses column (set by compute_pairs).
    """
    return headways.filter(pl.col("n_buses") >= min_buses)

## Module: preprocessing/pipeline

Orquestación gated por `centerline_strategy_for(empresaid)` (R-CFG1).
E4 → "single" (single-pass) PROVISIONALMENTE: sin evidencia multi-filar aún.

In [ ]:
"""Two-pass pipeline orchestration for multi-filar corridor empresas.

Provides:
  run_two_pass_pipeline — pass-1 single centerline → crude direction labels →
                          pass-2 per-direction centerlines → refined labels →
                          assign_trip_ids (R-PIPE1, R-PIPE2).
  run_single_pass_pipeline — the original single-pass path for backward
                             compatibility and as a discriminator for tests.

Both functions accept a pre-filtered empresa sub-frame (already empresa-filtered,
speed-attached with attach_observed_speed) and return a processed DataFrame.

Design: docs/decisiones-headway-fase2.md §8, SDD multi-filar-direction-balanced-
centerline design §4.
"""
from __future__ import annotations

import polars as pl



def run_two_pass_pipeline(
    gps: pl.DataFrame,
    *,
    empresaid: int,
    return_pass1_s: bool = False,
) -> pl.DataFrame | tuple[pl.DataFrame, pl.Series]:
    """Execute the two-pass centerline pipeline for a multi-filar empresa.

    Pass 1 (existing behavior):
        build_centerline → project_to_centerline → infer_direction

    Pass 2 (new, per-direction):
        build_centerline_per_direction → project_per_direction → infer_direction

    Then (R-PIPE2 invariant — ALWAYS after the second infer_direction):
        assign_trip_ids

    Args:
        gps: empresa-filtered GPS DataFrame with columns
             (empresaid, unidadid, time, lat, lon, speed_kmh).
        empresaid: which empresa is being processed.
        return_pass1_s: if True, return a tuple (result, pass1_s_snapshot) for
                        the s-continuity test (T3.2). Otherwise return result only.

    Returns:
        Processed pl.DataFrame with direction, s, lateral_m, trip_id columns.
        If return_pass1_s is True, returns (result, pass1_s_snapshot) instead.
    """
    # ---- Pass 1 ----
    cl_pass1 = build_centerline(gps, empresaid=empresaid)
    sub = project_to_centerline(gps, cl_pass1, empresaid=empresaid)
    sub = infer_direction(sub)

    # Snapshot pass-1 s for the continuity guard (spec Q-S-CONTINUITY)
    pass1_s_snapshot = sub["s"]

    # ---- Pass 2 ----
    cls = build_centerline_per_direction(
        sub,
        empresaid=empresaid,
        min_pings_per_dir=PRODUCTIVE_PARAMS.centerline_min_pings_per_direction,
    )
    sub = project_per_direction(sub, cls, empresaid=empresaid)
    sub = infer_direction(sub)

    # s-continuity runtime assertion (spec Q-S-CONTINUITY)
    assert "s" in sub.columns, (
        "s column missing after pass-2 projection (with_columns upsert failed)"
    )

    # ---- R-PIPE2: assign_trip_ids MUST run AFTER second infer_direction ----
    sub = assign_trip_ids(sub)

    if return_pass1_s:
        return sub, pass1_s_snapshot
    return sub


def run_single_pass_pipeline(
    gps: pl.DataFrame,
    *,
    empresaid: int,
) -> pl.DataFrame:
    """Execute the original single-pass centerline pipeline.

    Single pass:
        build_centerline → project_to_centerline → infer_direction → assign_trip_ids

    Used for empresas with centerline_strategy == "single" and as a discriminator
    baseline in the two-pass integration test (T3.3).

    Args:
        gps: empresa-filtered GPS DataFrame with speed_kmh already attached.
        empresaid: which empresa is being processed.

    Returns:
        Processed pl.DataFrame with direction, s, lateral_m, trip_id columns.
    """
    cl = build_centerline(gps, empresaid=empresaid)
    sub = project_to_centerline(gps, cl, empresaid=empresaid)
    sub = infer_direction(sub)
    sub = assign_trip_ids(sub)
    return sub

## Ejecutar pipeline de preprocessing — E4

Carga `clean_gps.parquet`, aplica todos los módulos en orden de dependencia
para `empresaid=4`, y escribe `headways_E4.parquet` (+ cleaned_gps, null_buckets).

La estrategia (single/two-pass) la decide `centerline_strategy_for(4)` — NO se
hardcodea. E4 devuelve "single" (single-pass).

In [ ]:

lf = (
    pl.scan_parquet(INPUT)
    .filter(
        pl.col("empresaid").is_in(EMPRESAS)
        & pl.col("time").is_not_null()
        & pl.col("lat").is_not_null() & pl.col("lon").is_not_null()
        & (pl.col("lat") != 0) & (pl.col("lon") != 0)
    )
    .with_columns(pl.col("time").dt.date().alias("day"))
    .sort(["empresaid", "unidadid", "time"])
)
gps_all = lf.collect(engine="streaming")
print(f"Rows loaded: {gps_all.height:,}")

for empresaid in EMPRESAS:
    print(f"\n--- Empresa {empresaid} ---")
    sub = gps_all.filter(pl.col("empresaid") == empresaid)

    sub = attach_observed_speed(sub)
    strategy = centerline_strategy_for(empresaid)

    if strategy == "two-pass":
        print(f"  Strategy: two-pass (multi-filar)")
        # Pass-1: single centerline → crude direction labels
        centerline = build_centerline(sub, empresaid=empresaid)
        sub = project_to_centerline(sub, centerline, empresaid=empresaid)
        sub = infer_direction(sub)
        pass1_s = sub["s"]  # snapshot for continuity assertion

        # Pass-2: per-direction centerlines → refined labels
        cls = build_centerline_per_direction(
            sub, empresaid=empresaid,
            min_pings_per_dir=PRODUCTIVE_PARAMS.centerline_min_pings_per_direction,
        )
        sub = project_per_direction(sub, cls, empresaid=empresaid)
        sub = infer_direction(sub)

        # s-continuity runtime assertion (spec Q-S-CONTINUITY)
        assert "s" in sub.columns, "s column missing after pass-2 projection"

        # R-PIPE2: assign_trip_ids MUST run AFTER second infer_direction
        sub = assign_trip_ids(sub)
    else:
        print(f"  Strategy: single-pass")
        centerline = build_centerline(sub, empresaid=empresaid)
        sub = project_to_centerline(sub, centerline, empresaid=empresaid)
        sub = infer_direction(sub)
        sub = assign_trip_ids(sub)

    snaps = build_snapshots(sub)
    heads, null_buckets = compute_headways_c2(snaps, sub)

    out_gps = OUTPUT_DIR / f"cleaned_gps_E{empresaid}.parquet"
    out_hw = OUTPUT_DIR / f"headways_E{empresaid}.parquet"
    out_buckets = OUTPUT_DIR / f"headway_null_buckets_E{empresaid}.parquet"
    sub.rename({"time": "t"}).select(
        ["unidadid", "t", "lat", "lon", "s", "direction", "speed_kmh", "lateral_m"]
    ).write_parquet(out_gps)
    heads.write_parquet(out_hw)
    null_buckets.write_parquet(out_buckets)

    print(f"  cleaned_gps:  {sub.height:,} rows → {out_gps}")
    print(f"  headways:     {heads.height:,} rows → {out_hw}")
    print(f"  non-null hw:  {heads.filter(pl.col('delta_t_min').is_not_null()).height:,}")
    print(f"  null_buckets: {null_buckets.height:,} rows → {out_buckets}")

## Auditoría de sanidad — E4

Verifica invariantes (INV-4/6/7/8) y la cobertura de `delta_t_min`. La tasa de
crossings stale aquí es la evidencia para decidir si E4 debe pasar a "two-pass"
(flip de 1 línea en `config.py`).

In [ ]:

for empresaid in EMPRESAS:
    out_gps = OUTPUT_DIR / f"cleaned_gps_E{empresaid}.parquet"
    out_hw = OUTPUT_DIR / f"headways_E{empresaid}.parquet"
    if not out_gps.exists() or not out_hw.exists():
        print(f"E{empresaid}: output files not found, skip audit")
        continue

    gps_e = pl.read_parquet(out_gps)
    hw_e = pl.read_parquet(out_hw)

    print(f"\n=== E{empresaid} audit ===")
    print(f"  cleaned_gps: {gps_e.height:,} rows, {gps_e.width} cols")
    print(f"  headways:    {hw_e.height:,} rows, {hw_e.width} cols")

    if hw_e.height > 0:
        bad_seconds = hw_e.filter(pl.col("t").dt.second() != 0).height
        print(f"  INV-6 violations (t.second != 0): {bad_seconds}")
        bad_n = hw_e.filter(pl.col("n_buses") < 2).height
        print(f"  INV-4 violations (n_buses < 2): {bad_n}")
        bad_pair = hw_e.filter(pl.col("bus_front") == pl.col("bus_back")).height
        print(f"  INV-7 violations (bus_front == bus_back): {bad_pair}")

    if gps_e.height > 0:
        bad_lat = gps_e.filter(pl.col("lateral_m") > 300.0).height
        print(f"  INV-8 violations (lateral_m > 300): {bad_lat}")

    if hw_e.height > 0:
        null_frac = hw_e.filter(pl.col("delta_t_min").is_null()).height / hw_e.height
        print(f"  delta_t_min null fraction: {null_frac:.1%}")
        print(f"  delta_t_min stats: {hw_e['delta_t_min'].drop_nulls().describe()}")

    if hw_e.height > 0:
        pairs_per_day = (
            hw_e.filter(pl.col("delta_t_min").is_not_null())
            .with_columns(pl.col("t").dt.date().alias("day"))
            .group_by("day").len().sort("day")
        )
        print(f"  pairs_efectivo/day: min={pairs_per_day['len'].min():,} "
              f"max={pairs_per_day['len'].max():,} mean={int(pairs_per_day['len'].mean()):,}")

## Baselines — embed de la librería de evaluación

A partir de aquí se reusa la librería de NB10 (sin tocarla) para evaluar los
baselines clásicos sobre las headways E4 recién calculadas.

## Module: evaluation/splits

Temporal split (`split_temporal`) y winsorización train-only p99
(`winsorize_train_p99`). Los rangos de fecha están fijados en spec §3.

In [ ]:
"""Temporal split and winsorization helpers for headway evaluation — Fase 3.

Public API:
    split_temporal(df: pl.DataFrame) -> pl.DataFrame
    winsorize_train_p99(df: pl.DataFrame) -> tuple[pl.DataFrame, float]

Constants (split date ranges, locked in spec §3 and design §5):
    SPLIT_TRAIN_START, SPLIT_TRAIN_END
    SPLIT_VAL_START,   SPLIT_VAL_END
    SPLIT_TEST_START,  SPLIT_TEST_END
    WINSOR_QUANTILE

Design decisions (locked in design §5 and §9):
  - Split key is pl.col("t").dt.date() membership, NOT row index.
  - Three ranges are exhaustive and mutually exclusive.
  - Rows outside all three ranges receive None (split column = null).
  - Winsorization threshold is computed on train rows only (AC-WINSOR-1, AC-WINSOR-2).
  - Null delta_t_min rows are NOT clipped (AC-WINSOR-3).
  - Rows above threshold are clipped (not dropped) (AC-WINSOR-4).
  - Constants live here (not PRODUCTIVE_PARAMS) — evaluation protocol concern.
  - WINSOR_QUANTILE and split dates are not added to pyproject.toml.
"""
from __future__ import annotations

from datetime import date

import polars as pl

# ---------------------------------------------------------------------------
# Split date range constants (spec §3, inclusive on both ends)
# ---------------------------------------------------------------------------

SPLIT_TRAIN_START: date = date(2023, 10, 1)
SPLIT_TRAIN_END:   date = date(2024, 1, 15)

SPLIT_VAL_START:   date = date(2024, 1, 16)
SPLIT_VAL_END:     date = date(2024, 2, 7)

SPLIT_TEST_START:  date = date(2024, 2, 8)
SPLIT_TEST_END:    date = date(2024, 2, 29)

WINSOR_QUANTILE: float = 0.99


def split_temporal(df: pl.DataFrame) -> pl.DataFrame:
    """Add a `split` column (Utf8) with values {"train", "val", "test"}.

    Membership is determined by pl.col("t").dt.date() against the six
    module-level date constants.  Rows outside all three ranges receive
    null (should not exist in the R7 v4 dataset; harness raises if found).

    Parameters
    ----------
    df:
        headways DataFrame containing at least a `t` (Datetime) column.

    Returns
    -------
    pl.DataFrame — input frame with one added column `split: Utf8`.
    """
    day = pl.col("t").dt.date()
    return df.with_columns(
        pl.when((day >= SPLIT_TRAIN_START) & (day <= SPLIT_TRAIN_END))
          .then(pl.lit("train"))
          .when((day >= SPLIT_VAL_START) & (day <= SPLIT_VAL_END))
          .then(pl.lit("val"))
          .when((day >= SPLIT_TEST_START) & (day <= SPLIT_TEST_END))
          .then(pl.lit("test"))
          .otherwise(None)
          .alias("split")
    )


def winsorize_train_p99(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float]:
    """Clip delta_t_min to the 99th-percentile threshold computed on train rows only.

    The threshold is computed once as a scalar from non-null train-split rows.
    It is then applied as a clip ceiling to ALL rows (train + val + test).
    Null delta_t_min values are never clipped — they remain null (AC-WINSOR-3).

    Parameters
    ----------
    df:
        headways DataFrame that already has a `split` column (added by
        split_temporal) and a `delta_t_min` (Float64 nullable) column.

    Returns
    -------
    (clipped_df, threshold)
        clipped_df: same schema as df, delta_t_min clipped.
        threshold: the scalar train-p99 value used as the clip ceiling.

    Design note (AC-WINSOR-2 leakage guard):
        The filter `split == "train"` is applied BEFORE computing the quantile,
        so extreme outliers in val or test rows cannot shift the threshold.
    """
    threshold = float(
        df.filter(
            (pl.col("split") == "train") & pl.col("delta_t_min").is_not_null()
        )["delta_t_min"]
        .quantile(WINSOR_QUANTILE)
    )

    # Clip: preserve null rows; clip non-null rows to threshold from above.
    # pl.min_horizontal(col, lit(threshold)) would coerce null → 0 in some
    # polars versions, so we use the explicit when/then pattern (design §5).
    clipped = df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
          .then(None)
          .otherwise(
              pl.min_horizontal(pl.col("delta_t_min"), pl.lit(threshold))
          )
          .alias("delta_t_min")
    )
    return clipped, threshold

## Module: evaluation/metrics

`mae` y `rmse` en minutos. Aceptan polars Series o numpy. Filas null/NaN se
descartan antes de agregar. MAPE excluido (spec B3-NO-MAPE).

In [ ]:
"""Evaluation metrics for headway forecasting — Fase 3.

Public API:
    mae(y_true, y_pred) -> float
    rmse(y_true, y_pred) -> float

Both functions accept polars Series (Float64) or numpy arrays (float64).
Null / NaN masking: rows where EITHER y_true or y_pred is null/NaN are
dropped before aggregation.  If no valid rows remain, ValueError is raised.

Design decisions locked in design §4:
  - ValueError on empty/all-null input (NOT silent NaN return).
  - Only MAE and RMSE are in scope (spec B3-NO-MAPE — ratio-based metrics
    are out of scope because near-zero headways cause denominator blow-up).
  - No new pyproject.toml dependencies (polars + numpy already present).
"""
from __future__ import annotations

import numpy as np
import polars as pl


def _to_numpy_with_mask(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Coerce both inputs to float64 numpy arrays and apply the null/NaN mask.

    Polars Series with dtype Float64: null cells become NaN via .to_numpy().
    numpy arrays: assumed to already use NaN for missing values.

    Returns
    -------
    (y_true_masked, y_pred_masked) — two 1-D float64 arrays of equal length,
    containing no NaN values.  May be empty if all rows were masked.
    """
    # Coerce to numpy.
    if isinstance(y_true, pl.Series):
        yt = y_true.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yt = np.asarray(y_true, dtype=np.float64).ravel()

    if isinstance(y_pred, pl.Series):
        yp = y_pred.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yp = np.asarray(y_pred, dtype=np.float64).ravel()

    # Elementwise mask: keep row only if BOTH sides are finite (not NaN).
    mask = ~(np.isnan(yt) | np.isnan(yp))
    return yt[mask], yp[mask]


def mae(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Mean Absolute Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — MAE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "mae: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.mean(np.abs(yt - yp)))


def rmse(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Root Mean Squared Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — RMSE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "rmse: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.sqrt(np.mean((yt - yp) ** 2)))

## Module: baselines/statistical

B0 media global, B1 persistencia naive (horizon-aware), B2 media móvil
(w∈{5,10,15}, horizon-aware), B3 suavizado exponencial (α=0.3, horizon-aware),
B4 promedio histórico por hora. Operan por slot `(empresaid, direction, pair_rank)`.

In [ ]:
"""Classical statistical baselines for headway forecasting — Fase 3.

Public API:
    predict_b0(headways: pl.DataFrame) -> pl.DataFrame
    predict_b1(headways: pl.DataFrame, *, horizon: int = 1) -> pl.DataFrame
    predict_b2(headways: pl.DataFrame, *, window: int, horizon: int = 1) -> pl.DataFrame
    predict_b3(headways: pl.DataFrame, *, alpha: float = SES_ALPHA, horizon: int = 1) -> pl.DataFrame
    predict_b4_ha(headways: pl.DataFrame) -> pl.DataFrame

Input contract (all four functions):
    The DataFrame must have a `split` column (Utf8) added by split_temporal.
    Columns consumed: empresaid, t, direction, pair_rank, delta_t_min, split.

Output contract:
    Each function returns the input frame with ONE additional column:
        B0 → y_pred_b0
        B1 → y_pred_b1
        B2 → y_pred_b2_w{window}
        B3 → y_pred_b3

Predictions are filled for ALL rows (train + test); the evaluation harness
consumes test rows only.  Filling train rows costs negligibly more and lets
future SDDs reuse predictions if needed (design §3).

Design decisions locked in design §3 and §9:
  - Functions, not classes (consistent with project precedent).
  - Slot key: (empresaid, direction, pair_rank).
  - B2 `window` = count of last NON-NULL observations (not a time window).
  - B2 min_periods = window // 2  (floor division).
  - B3 alpha = SES_ALPHA = 0.3, per-slot online recursion, null-skip.
  - B3 state init: NaN until first non-null train obs; first non-null sets s directly.
  - No new pyproject.toml dependencies (polars + numpy only).
"""
from __future__ import annotations

import numpy as np
import polars as pl

# ---------------------------------------------------------------------------
# Module-level constants (locked in design §9)
# ---------------------------------------------------------------------------

BASELINE_B2_WINDOWS: tuple[int, ...] = (5, 10, 15)
SES_ALPHA: float = 0.3

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


# ===========================================================================
# B0 — Global mean per slot (train rows only)
# ===========================================================================

def predict_b0(headways: pl.DataFrame) -> pl.DataFrame:
    """Add column `y_pred_b0`: per-slot mean of train delta_t_min.

    The prediction is constant within a slot — the arithmetic mean of all
    non-null delta_t_min values in the train split for that slot.  Slots
    with no non-null train observations receive null (AC-B0-2).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b0` (Float64 nullable) added.
    """
    train_means = (
        headways
        .filter(pl.col("split") == "train")
        .group_by(_SLOT_COLS)
        .agg(pl.col("delta_t_min").mean().alias("y_pred_b0"))
    )
    return headways.join(train_means, on=_SLOT_COLS, how="left")


# ===========================================================================
# B1 — Naive / persistence baseline
# ===========================================================================

def predict_b1(headways: pl.DataFrame, *, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b1`: last non-null delta_t_min seen `horizon` steps before each row.

    Uses forward_fill().shift(horizon).over(slot) — the canonical polars pattern for
    ŷ_{t+h} = y_t with null gaps.  Causal by construction (shift prevents the
    current-row value from appearing as its own prediction).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    horizon:
        Number of steps to shift. Default 1 reproduces the original behavior.
        Calling ``predict_b1(df)`` (no horizon arg) is identical to ``predict_b1(df, horizon=1)``.

    Returns
    -------
    pl.DataFrame — input frame sorted by (slot, t), with `y_pred_b1` added.
    """
    return (
        headways
        .sort(_SLOT_COLS + ["t"])
        .with_columns(
            pl.col("delta_t_min")
              .forward_fill()
              .shift(horizon)
              .over(_SLOT_COLS)
              .alias("y_pred_b1")
        )
    )


# ===========================================================================
# B2 — Trailing moving average of last w NON-NULL observations
# ===========================================================================

def predict_b2(headways: pl.DataFrame, *, window: int, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b2_w{window}`: mean of last `window` non-null observations.

    Semantics (locked, design §3):
      - `window` is a COUNT of non-null observations, not a time window.
      - min_periods = window // 2  (floor).
      - Prediction at row i uses only observations strictly before row i (causal).

    Horizon rule (Fase 6.5):
      The 1-step prediction is computed first (rolling_mean.shift(1) on the
      non-null sub-series + join_asof backward).  Then shift(horizon-1) is
      applied over the slot key so that ŷ_{t+h} = ŷ_{t+1} lagged by h-1
      additional steps.  horizon=1 → shift(0) = identity (backward-compatible).

    Implementation:
      - group_by(slot).map_groups(lambda g: _b2_one_slot(g, window))
      - Within each group: extract non-null values, compute rolling_mean with
        shift(1) (causal), then join_asof(strategy="backward") back to
        original group rows on `t`.
      - After concat, apply shift(horizon-1).over(_SLOT_COLS).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    window:
        Number of non-null observations in the trailing window.
    horizon:
        Prediction horizon in steps.  Default 1 reproduces the original
        behavior exactly.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b2_w{window}` (Float64 nullable) added.
    """
    col_name = f"y_pred_b2_w{window}"
    min_periods = window // 2

    def _b2_one_slot(group: pl.DataFrame) -> pl.DataFrame:
        group = group.sort("t")

        # Extract non-null rows only, in time order.
        non_null = group.filter(pl.col("delta_t_min").is_not_null())

        if len(non_null) == 0:
            # No non-null observations: all predictions are null.
            return group.with_columns(pl.lit(None, dtype=pl.Float64).alias(col_name))

        # Compute rolling mean on the non-null sub-series, then shift(1) for
        # causality: the prediction at position i uses observations 0..i-1.
        non_null = non_null.with_columns(
            pl.col("delta_t_min")
              .rolling_mean(window_size=window, min_samples=min_periods)
              .shift(1)
              .alias(col_name)
        )

        # Align back to the full group (including null rows) via join_asof.
        # strategy="backward" finds the most recent non-null rolling mean at or
        # before each timestamp in the original group.
        result = group.join_asof(
            non_null.select(["t", col_name]),
            on="t",
            strategy="backward",
        )
        return result

    sorted_df = headways.sort(_SLOT_COLS + ["t"])
    slots = sorted_df.partition_by(_SLOT_COLS, maintain_order=True)
    result = pl.concat([_b2_one_slot(g) for g in slots])

    # Apply horizon shift: shift(horizon-1) over slot so predictions look
    # h steps ahead.  shift(0) is a no-op → backward-compatible for horizon=1.
    if horizon > 1:
        result = (
            result
            .sort(_SLOT_COLS + ["t"])
            .with_columns(
                pl.col(col_name)
                  .shift(horizon - 1)
                  .over(_SLOT_COLS)
                  .alias(col_name)
            )
        )

    return result


# ===========================================================================
# B3 — Simple Exponential Smoothing (α=0.3, per slot, online)
# ===========================================================================

def _ses_one_slot(slot_df: pl.DataFrame, alpha: float) -> pl.DataFrame:
    """Online SES recursion for a single slot.

    s_t = α·y_t + (1-α)·s_{t-1}  (null observations skip the update).
    pred[i] = s before observing y[i]  (causal: shift-1 semantics).

    Initialization: s = NaN until the first non-null y_t; the first non-null
    value sets s directly (no prior needed) — AC-B3-3.  The prediction at
    that initialization row is NaN (no prior state), so the first test
    prediction for a slot with at least one train observation is the state
    after consuming ALL train rows.
    """
    slot_df = slot_df.sort("t")
    y = slot_df["delta_t_min"].to_numpy(allow_copy=True).astype(np.float64)
    pred = np.full(len(y), np.nan)
    s = np.nan  # smoothing state; NaN until first non-null

    for i in range(len(y)):
        # Prediction at row i is the state BEFORE observing y[i].
        pred[i] = s
        # Update state if current observation is not null/NaN.
        if not np.isnan(y[i]):
            if np.isnan(s):
                s = y[i]  # initialization: first non-null sets state directly
            else:
                s = alpha * y[i] + (1.0 - alpha) * s

    # Convert float NaN → polars null so downstream is_null() works correctly.
    pred_series = pl.Series("y_pred_b3", pred, dtype=pl.Float64)
    return slot_df.with_columns(pred_series.set(pred_series.is_nan(), None))


def predict_b3(headways: pl.DataFrame, *, alpha: float = SES_ALPHA, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b3`: online SES predictions, α=0.3 (default).

    Applies the causal recursion s_t = α·y_t + (1-α)·s_{t-1} per slot.
    Null observations do not update the state (AC-B3-2).
    State is initialized from the first non-null observation (AC-B3-3).
    Slots with all-null values emit null for all rows (AC-B3-4).

    Horizon rule (Fase 6.5):
      The 1-step SES predictions are computed first (existing per-slot loop).
      Then shift(horizon-1) is applied over the slot key so that ŷ_{t+h} =
      ŷ_{t+1} lagged by h-1 additional steps.
      horizon=1 → shift(0) = identity (backward-compatible).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    alpha:
        Smoothing parameter.  Default SES_ALPHA = 0.3 (locked, design §3).
        Tests may pass alternative values for edge-case verification.
    horizon:
        Prediction horizon in steps.  Default 1 reproduces the original
        behavior exactly.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b3` (Float64 nullable) added.

    Design note (D-PL-OVER-VS-MAPGROUPS):
        polars .over() does not support stateful numpy loops; map_groups
        materializes one Python frame per slot (~30–100 per corridor — trivially fast).
    """
    sorted_df = headways.sort(_SLOT_COLS + ["t"])
    slots = sorted_df.partition_by(_SLOT_COLS, maintain_order=True)
    result = pl.concat([_ses_one_slot(g, alpha) for g in slots])

    # Apply horizon shift: shift(horizon-1) over slot so predictions look
    # h steps ahead.  shift(0) is a no-op → backward-compatible for horizon=1.
    if horizon > 1:
        result = (
            result
            .sort(_SLOT_COLS + ["t"])
            .with_columns(
                pl.col("y_pred_b3")
                  .shift(horizon - 1)
                  .over(_SLOT_COLS)
                  .alias("y_pred_b3")
            )
        )

    return result


# ===========================================================================
# B4 — Historical Average per (slot, hour-of-day) from train only
# ===========================================================================

def predict_b4_ha(headways: pl.DataFrame) -> pl.DataFrame:
    """Add column `y_pred_b4_ha`: per-slot, per-hour mean of train delta_t_min.

    For each (empresaid, direction, pair_rank, hour) group, computes the mean
    of non-null delta_t_min values from train rows only.  Test/val rows at the
    same hour receive that mean as their prediction.  Hours not seen in train
    produce null predictions.
    """
    ha_key = _SLOT_COLS + ["_hour"]
    df = headways.with_columns(pl.col("t").dt.hour().alias("_hour"))

    train_means = (
        df
        .filter(pl.col("split") == "train")
        .group_by(ha_key)
        .agg(pl.col("delta_t_min").mean().alias("y_pred_b4_ha"))
    )

    result = df.join(train_means, on=ha_key, how="left")
    return result.drop("_hour")

## Module: baselines/fitted

`predict_b5_xgb` — baseline ajustado (B5_XGB) sobre 12 lags + features de
calendario/slot. Debe embeberse ANTES de harness, que lo llama.

In [ ]:
"""Fitted ML baseline for headway forecasting — gradient-boosted regressor (B5_XGB).

Why this module is separate from `statistical.py`:
    B0-B4 are closed-form/recursive predictors with NO learned parameters and a
    "no new dependencies" design lock. B5_XGB is a *fitted* learner (XGBoost) —
    a different category. It answers the reviewer reflex "where is a fitted/ML
    baseline?" that pure naive baselines (persistence, moving average, SES,
    historical average) do not.

Design — fair comparison to the DL models (NB11-13):
    The DL models consume an input window of T_in = 12 consecutive 1-minute
    steps and predict the headway HORIZON steps after the last input step. The
    XGBoost baseline is given the SAME information: 12 lagged headway values
    ending HORIZON steps before the target, so `lag_1` equals the B1 persistence
    prediction (`shift(horizon)`) and the model strictly extends the naive
    baselines rather than seeing extra future data. Calendar context (hour,
    weekday) and static slot keys (direction, pair_rank) round out the features.

Contract (mirrors statistical.py):
    predict_b5_xgb(headways, *, horizon=1, seed=42) -> headways + y_pred_b5_xgb
    Input must have the `split` column (added by split_temporal). The model is
    fit on TRAIN rows only; predictions are produced for ALL rows. Validation
    rows are used for early stopping ONLY when there are enough of them
    (>= _MIN_VAL_ROWS); otherwise a fixed number of trees is used.

Determinism:
    Single-threaded (`n_jobs=1`), fixed `random_state`, `tree_method="hist"` →
    repeated calls on the same machine produce identical predictions.
"""
from __future__ import annotations

import numpy as np
import polars as pl

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]

# Number of lagged headway steps fed to the model = DL input window (T_in).
N_LAGS: int = 12

# Use validation rows for early stopping only when there are at least this many;
# tiny test fixtures (and corridors with no val rows) fall back to fixed trees.
_MIN_VAL_ROWS: int = 50

# Fixed gradient-boosting hyperparameters (native xgboost API, no sklearn dep).
# Deliberately modest and regularized: a credible fitted competitor, not an
# over-tuned one. nthread=1 + fixed seed + hist tree method → deterministic.
_NUM_BOOST_ROUND: int = 400
_EARLY_STOPPING_ROUNDS: int = 30

_XGB_PARAMS: dict = {
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "lambda": 1.0,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "nthread": 1,
}


def _build_features(headways: pl.DataFrame, *, horizon: int) -> tuple[pl.DataFrame, list[str]]:
    """Return the frame sorted by (slot, t) with lag + calendar feature columns
    added, plus the list of feature column names.

    lag_k (k = 1..N_LAGS) = headway value (forward-filled within slot) observed
    `horizon + k - 1` steps before the target row. lag_1 == B1 persistence.
    """
    lag_exprs = [
        pl.col("delta_t_min")
        .forward_fill()
        .shift(horizon + k - 1)
        .over(_SLOT_COLS)
        .alias(f"_lag_{k}")
        for k in range(1, N_LAGS + 1)
    ]
    df = (
        headways
        .sort(_SLOT_COLS + ["t"])
        .with_columns(
            *lag_exprs,
            pl.col("t").dt.hour().alias("_hour"),
            pl.col("t").dt.weekday().alias("_weekday"),
        )
    )
    feature_cols = (
        [f"_lag_{k}" for k in range(1, N_LAGS + 1)]
        + ["_hour", "_weekday", "direction", "pair_rank"]
    )
    return df, feature_cols


def predict_b5_xgb(
    headways: pl.DataFrame,
    *,
    horizon: int = 1,
    seed: int = 42,
) -> pl.DataFrame:
    """Add column `y_pred_b5_xgb`: gradient-boosted forecast of delta_t_min.

    Parameters
    ----------
    headways:
        headways DataFrame with the `split` column attached. Columns consumed:
        empresaid, t, direction, pair_rank, delta_t_min, split.
    horizon:
        Forecast horizon in steps. lag_1 = shift(horizon) so the 1-lag feature
        equals B1 persistence; horizon=1 is the default.
    seed:
        Random seed for reproducibility.

    Returns
    -------
    pl.DataFrame — input frame (sorted by slot, t) with `y_pred_b5_xgb`
        (Float64 nullable) added. If the train split has no usable rows, the
        column is all-null.
    """
    import xgboost as xgb

    original_cols = headways.columns
    df, feature_cols = _build_features(headways, horizon=horizon)

    is_train = df["split"] == "train"
    is_val = df["split"] == "val"
    target_present = df["delta_t_min"].is_not_null()

    train_mask = (is_train & target_present).to_numpy()
    n_train = int(train_mask.sum())

    # Degenerate: nothing to fit on → null predictions (mirrors B0 on empty slots).
    if n_train == 0:
        return df.select(original_cols).with_columns(
            pl.lit(None, dtype=pl.Float64).alias("y_pred_b5_xgb")
        )

    X_all = df.select(feature_cols).to_numpy().astype(np.float64)
    y_all = df["delta_t_min"].to_numpy().astype(np.float64)

    dtrain = xgb.DMatrix(X_all[train_mask], label=y_all[train_mask], missing=np.nan)
    dall = xgb.DMatrix(X_all, missing=np.nan)

    params = dict(_XGB_PARAMS, seed=seed)

    val_mask = (is_val & target_present).to_numpy()
    if int(val_mask.sum()) >= _MIN_VAL_ROWS:
        # Use validation for early stopping (the DL models also tuned on val).
        dval = xgb.DMatrix(X_all[val_mask], label=y_all[val_mask], missing=np.nan)
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=_NUM_BOOST_ROUND,
            evals=[(dval, "val")],
            early_stopping_rounds=_EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
    else:
        booster = xgb.train(params, dtrain, num_boost_round=_NUM_BOOST_ROUND)

    preds = booster.predict(dall).astype(np.float64)

    return df.select(original_cols).with_columns(
        pl.Series("y_pred_b5_xgb", preds, dtype=pl.Float64)
    )

## Module: baselines/harness

`evaluate_corridor` compone split → winsorize → B0-B4 + B5_XGB
(include_fitted=True por defecto) → métricas por (direction × baseline),
devolviendo un DataFrame long de 48 filas. Acepta `horizon`.

In [ ]:
"""Evaluation harness for classical baseline comparison — Fase 3.

Public API:
    evaluate_corridor(headways: pl.DataFrame, corridor_name: str) -> pl.DataFrame

The function composes the full pipeline for one corridor:
    split_temporal → winsorize_train_p99 → predict_b0/b1/b2(×3)/b3/b4_ha
    [→ predict_b5_xgb when include_fitted]
    → filter test rows → compute MAE + RMSE per (direction, baseline)
    → return tidy long-form DataFrame.

Output schema (design §6):
    corridor   Utf8
    direction  Utf8   — "-1", "+1", "aggregate"
    baseline   Utf8   — "B0", "B1", "B2_w5", "B2_w10", "B2_w15", "B3", "B4_HA"
                        [, "B5_XGB" when include_fitted]
    metric     Utf8   — "MAE", "RMSE"
    value      Float64 — minutes

Rows per corridor: 3 directions × N baselines × 2 metrics.
    include_fitted=True  (default): N = 8 → 48 rows per corridor.
    include_fitted=False (formulaic-only): N = 7 → 42 rows per corridor.

Design decisions (locked in design §6 and §9):
  - "aggregate" direction = MAE/RMSE over POOLED test rows (both directions
    concatenated), NOT mean of per-direction metrics.
  - val rows are NEVER consumed by the formulaic baselines B0-B4 (B3-VAL-UNUSED).
    The fitted baseline B5_XGB MAY use val rows for early stopping (only when
    there are enough), which is correct practice for a learned model and mirrors
    how the DL models were tuned.
  - B5_XGB (the fitted ML baseline) adds an xgboost dependency; it lives in
    fitted.py and is opt-out via include_fitted=False.
  - harness.py does NOT read parquets or write CSV (notebook does those).
"""
from __future__ import annotations

import polars as pl


# Map from prediction column name → display name for the output DataFrame.
_BASELINE_MAP: list[tuple[str, str]] = [
    ("y_pred_b0", "B0"),
    ("y_pred_b1", "B1"),
    ("y_pred_b2_w5", "B2_w5"),
    ("y_pred_b2_w10", "B2_w10"),
    ("y_pred_b2_w15", "B2_w15"),
    ("y_pred_b3", "B3"),
    ("y_pred_b4_ha", "B4_HA"),
]

# The fitted ML baseline is appended only when include_fitted=True.
_FITTED_ENTRY: tuple[str, str] = ("y_pred_b5_xgb", "B5_XGB")


def evaluate_corridor(
    headways: pl.DataFrame,
    corridor_name: str,
    *,
    horizon: int = 1,
    include_fitted: bool = True,
) -> pl.DataFrame:
    """Run all classical baselines on one corridor and return a tidy metrics table.

    Parameters
    ----------
    headways:
        Raw headways DataFrame with R7 v4 schema columns:
        empresaid, t, direction, pair_rank, delta_t_min.
        Must NOT already have a `split` column (this function adds it).
    corridor_name:
        Label for the `corridor` column in the output (e.g. "E2", "E59").
    horizon:
        Prediction horizon in steps. Default 1 reproduces the original behavior
        exactly. B1, B2, B3, and B5_XGB are horizon-aware and receive this value.
        B0 and B4_HA are horizon-agnostic (constant/lookup predictors) and are
        not affected.
    include_fitted:
        When True (default), also runs the fitted ML baseline B5_XGB (xgboost).
        When False, only the formulaic baselines B0-B4 run (no xgboost import).

    Returns
    -------
    pl.DataFrame — tidy long-form table (48 rows with include_fitted, else 42):
        [corridor, direction, baseline, metric, value]

    Notes
    -----
    - val rows are ignored at prediction time (baselines consume train only)
      and are never included in metric computation (metrics use test rows only).
    - The "aggregate" direction row pools test rows from both directions before
      computing MAE/RMSE — it is NOT the mean of the two per-direction metrics.
    """
    # --- Pipeline: split → winsorize → all baselines ---
    df = split_temporal(headways)
    df, _threshold = winsorize_train_p99(df)

    df = predict_b0(df)
    df = predict_b1(df, horizon=horizon)
    for w in BASELINE_B2_WINDOWS:
        df = predict_b2(df, window=w, horizon=horizon)
    df = predict_b3(df, horizon=horizon)
    df = predict_b4_ha(df)

    baseline_map = list(_BASELINE_MAP)
    if include_fitted:
        df = predict_b5_xgb(df, horizon=horizon)
        baseline_map = baseline_map + [_FITTED_ENTRY]

    # --- Filter to test rows only (B3-VAL-UNUSED) ---
    test_df = df.filter(pl.col("split") == "test")

    # --- Compute metrics per (direction × baseline) ---
    rows: list[dict] = []

    for pred_col, baseline_name in baseline_map:
        for direction_val in (-1, 1, "aggregate"):
            if direction_val == "aggregate":
                # Pool all test rows regardless of direction.
                subset = test_df
                direction_str = "aggregate"
            else:
                subset = test_df.filter(pl.col("direction") == direction_val)
                direction_str = f"+{direction_val}" if direction_val > 0 else str(direction_val)

            y_true = subset["delta_t_min"]
            y_pred = subset[pred_col]

            # Compute MAE and RMSE (null rows are masked inside the functions).
            mae_val = mae(y_true, y_pred)
            rmse_val = rmse(y_true, y_pred)

            rows.append(
                {
                    "corridor": corridor_name,
                    "direction": direction_str,
                    "baseline": baseline_name,
                    "metric": "MAE",
                    "value": mae_val,
                }
            )
            rows.append(
                {
                    "corridor": corridor_name,
                    "direction": direction_str,
                    "baseline": baseline_name,
                    "metric": "RMSE",
                    "value": rmse_val,
                }
            )

    return pl.DataFrame(rows).with_columns(
        pl.col("corridor").cast(pl.Utf8),
        pl.col("direction").cast(pl.Utf8),
        pl.col("baseline").cast(pl.Utf8),
        pl.col("metric").cast(pl.Utf8),
        pl.col("value").cast(pl.Float64),
    )

## Cargar headways E4 (calculadas en este mismo kernel)

Lee `headways_E4.parquet` recién escrito por la sección de preprocessing.
Inyecta `empresaid=4` como columna literal (el contrato del slot lo requiere).

In [ ]:

hw_e4 = pl.read_parquet(OUTPUT_DIR / "headways_E4.parquet").with_columns(
    pl.lit(4, dtype=pl.Int64).alias("empresaid")
)
print(f"E4: {hw_e4.height:,} rows, {hw_e4.width} cols")

# Cobertura por dirección (riesgo R-DIR1-COVERAGE).
summary = (
    hw_e4.group_by(["empresaid", "direction"])
    .agg([
        pl.len().alias("n_rows"),
        (pl.col("delta_t_min").is_not_null().sum() / pl.len()).alias("non_null_frac"),
    ])
    .sort(["empresaid", "direction"])
)
print(summary)

## Ejecutar harness — loop multi-horizonte (E4)

Llama a `evaluate_corridor(hw_e4, "E4", horizon=h)` para h ∈ {1, 3, 5, 10}
y concatena agregando la columna `horizon`. Salida esperada:
**192 filas = 4 horizontes × 1 corredor × 48** (3 dir × 8 baselines × 2 métricas).

In [ ]:

HORIZONS = [1, 3, 5, 10]
frames = []
for h in HORIZONS:
    r_e4 = evaluate_corridor(hw_e4, "E4", horizon=h)
    frames.append(r_e4.with_columns(pl.lit(h, dtype=pl.Int64).alias("horizon")))
results = pl.concat(frames)
print(f"Total rows: {results.height}  (expected 192 = 4 horizons x 1 corridor x 48)")
print(results.head(10))

## Escribir CSV — baselines_E4_results_multih.csv

Escribe la tabla long-form a `/kaggle/working/baselines_E4_results_multih.csv`.
Columnas: `corridor, direction, baseline, metric, value, horizon`.

In [ ]:

results.write_csv(CSV_OUT)
print(f"CSV written to: {CSV_OUT}")
print(f"Rows: {results.height}  Columns: {results.columns}")

## Tabla resumen (wide format)

Pivote ancho para lectura humana: filas = (corridor, horizon, direction, metric),
columnas = baseline.

In [ ]:

wide = results.pivot(
    on="baseline",
    index=["corridor", "horizon", "direction", "metric"],
    values="value",
)
print(wide.sort(["corridor", "horizon", "direction", "metric"]))